# CryoET 3D Segmentation — Self-contained PyTorch (Kaggle 2× T4)

Every line of the model/training/inference code lives **inline in this notebook**.
No external package imports, no `%%writefile`, no repo clone needed. Just run all cells.

**Pipeline:**
1. Define config + all PyTorch modules (config, data, models, losses, metrics, train, inference).
2. Set up the copick project + overlay (Stage 0).
3. Build sphere segmentation targets from ground-truth picks (Stage 1).
4. Train a 3D residual U-Net with AMP + class-balanced bootstrap sampling (Stage 2).
5. Sliding-window inference producing **raw labelmaps** (+ optional scoremaps) (Stage 3).
6. (Optional) Localize particles to coordinates (Stage 4).

**Hardware:** tuned for Kaggle 2× T4 (15 GB VRAM each), mixed precision fp16.
**Output:** raw segmentations as OME-Zarr in the copick overlay — postprocess yourself.

**Auto-resume:** the training loop automatically loads `net_weights_BEST.pt` (or
`net_weights_LAST.pt` as fallback) if present in the output dir, restoring model +
optimizer + scheduler + epoch + best-F1, so you can stop and re-run the training cell
without losing progress. Checkpoints are always saved/loaded unwrapped (no `module.`
prefix) so resume works correctly even with `DataParallel` across 2 GPUs.

Every epoch renders a side-by-side **full-slice** figure on one **validation tomogram**
(like sections 4a/4b): the model is tiled across the entire central XY slice and the
prediction is overlaid on the full tomogram slice next to the ground truth, displayed
**inline** under the training cell (and saved to `train_results/viz/epoch_NNN.png`).
Rich visualizations are interleaved throughout.

In [ ]:
# Install dependencies (idempotent). Kaggle images already have torch/numpy/scipy.
import sys, subprocess
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
pip_install("copick", "copick-utils", "ome-zarr", "tensorboard", "pyyaml", "tqdm", "matplotlib")
print("Deps installed.")

In [ ]:
import os, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")
assert torch.cuda.is_available(), "Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2"

WORK = "/kaggle/working"
INPUT = "/kaggle/input/competitions/czii-cryo-et-object-identification"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("Working dir:", os.getcwd())

## 1. Inline PyTorch modules

All code that would normally live in `backend/*.py` is defined here in the notebook
global namespace. Run cells top-to-bottom; later cells reference earlier names directly.

### Utilities: seeding, logging, checkpointing

In [ ]:
"""Reproducibility helpers."""

import os
import random
import numpy as np


def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:  # torch not installed (e.g. for utils-only use)
        pass


def worker_init_fn(worker_id: int) -> None:
    """Per-worker numpy/random seed for DataLoader workers."""
    import numpy as np
    import random

    seed = (np.random.get_state()[1][0] + worker_id) % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)

"""Lightweight logging."""

import logging
import sys

_FORMAT = "%(asctime)s | %(levelname)-7s | %(name)s | %(message)s"
_configured = False


def _configure() -> None:
    global _configured
    if _configured:
        return
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(_FORMAT, datefmt="%H:%M:%S"))
    root = logging.getLogger()
    root.addHandler(handler)
    root.setLevel(logging.INFO)
    _configured = True


def get_logger(name: str = "backend") -> logging.Logger:
    _configure()
    return logging.getLogger(name)

"""Checkpoint helpers."""

from pathlib import Path
from typing import Optional, Dict, Any
import torch


def _unwrap(model: torch.nn.Module) -> torch.nn.Module:
    """Return the underlying model, unwrapping DataParallel/DistributedDataParallel."""
    while isinstance(model, (torch.nn.DataParallel, torch.nn.parallel.DistributedDataParallel)):
        model = model.module
    return model


def _reconcile_state_dict(state_dict: Dict[str, Any], model: torch.nn.Module) -> Dict[str, Any]:
    """Match a checkpoint state_dict's key prefix to the model's wrapping.

    Saves always store the *unwrapped* (no `module.` prefix) state_dict, so we only
    need to add `module.` here when the target model is DataParallel-wrapped.
    """
    is_wrapped = isinstance(model, (torch.nn.DataParallel, torch.nn.parallel.DistributedDataParallel))
    sd_has_module = any(k.startswith("module.") for k in state_dict)

    if is_wrapped and not sd_has_module:
        # checkpoint is raw, model is wrapped -> add module. prefix
        return {"module." + k: v for k, v in state_dict.items()}
    if not is_wrapped and sd_has_module:
        # checkpoint is wrapped, model is raw -> strip module. prefix
        return {k[len("module."):]: v for k, v in state_dict.items() if k.startswith("module.")}
    return state_dict


def save_checkpoint(
    path: str,
    model: torch.nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    scaler: Optional[Any] = None,
    epoch: int = 0,
    best_metric: Optional[float] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> None:
    """Save a checkpoint. The model state_dict is always stored *unwrapped*
    (no `module.` prefix), so it can be loaded into either a raw or DataParallel-wrapped model."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    raw_model = _unwrap(model)
    state = {
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": epoch,
        "best_metric": best_metric,
        "extra": extra or {},
    }
    torch.save(state, path)


def load_checkpoint(
    path: str,
    model: torch.nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    scaler: Optional[Any] = None,
    map_location: str = "cpu",
) -> Dict[str, Any]:
    """Load a checkpoint, reconciling the `module.` prefix so loading works whether
    the target model is raw or DataParallel/DistributedDataParallel-wrapped."""
    ckpt = torch.load(path, map_location=map_location)
    state_dict = _reconcile_state_dict(ckpt["model"], model)
    model.load_state_dict(state_dict)
    if optimizer is not None and ckpt.get("optimizer") is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and ckpt.get("scheduler") is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    if scaler is not None and ckpt.get("scaler") is not None:
        scaler.load_state_dict(ckpt["scaler"])
    return ckpt

### Settings (typed YAML config)

In [ ]:
"""Typed configuration loaded from YAML."""


from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, List, Optional, Union

import yaml


def _as(dataclass_cls):
    """Decorator not needed; kept simple with explicit from_dict methods."""
    return dataclass_cls


# ---------------------------------------------------------------------------
# Section dataclasses
# ---------------------------------------------------------------------------
@dataclass
class DataCfg:
    copick_config: str = "/kaggle/working/copick.config"
    input_root: str = "/kaggle/input/competitions/czii-cryo-et-object-identification"
    output_root: str = "/kaggle/working"
    voxel_size: float = 10.0
    tomo_algorithm: str = "denoised"
    tomo_type: str = "denoised"
    target_name: str = "pytargets"
    target_user_id: str = "pytorch"
    target_session_id: str = "0"
    dim_in: int = 72
    l_rnd: int = 15
    background_ratio: float = 0.30
    sample_size: int = 5
    n_sub_epoch: int = 10
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    train_tomo_ids: Optional[List[str]] = None
    valid_tomo_ids: Optional[List[str]] = None

    @classmethod
    def from_dict(cls, d: dict) -> "DataCfg":
        known = {f for f in cls.__dataclass_fields__}
        return cls(**{k: v for k, v in d.items() if k in known})


@dataclass
class ModelCfg:
    name: str = "res_unet"
    filters: List[int] = field(default_factory=lambda: [48, 64, 128])
    dropout: float = 0.0
    in_channels: int = 1
    n_class: int = 8

    @classmethod
    def from_dict(cls, d: dict) -> "ModelCfg":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


@dataclass
class LossCfg:
    type: str = "ce_tversky"
    alpha: float = 0.3
    beta: float = 0.7
    gamma: float = 2.0
    ce_weight: float = 1.0
    tversky_weight: float = 1.0
    smooth: float = 1.0e-3
    class_weights: Union[str, List[float]] = "inverse"
    ignore_index: int = -1
    # Classes that carry target signal. If None, train() will default to
    # background + particle classes (membrane excluded until enabled).
    active_classes: Optional[List[int]] = None

    @classmethod
    def from_dict(cls, d: dict) -> "LossCfg":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


@dataclass
class TrainCfg:
    epochs: int = 70
    steps_per_epoch: int = 150
    batch_size: int = 8
    steps_per_valid: int = 20
    n_workers: int = 0  # avoid duplicating in-RAM tomogram pool across forked workers
    pin_memory: bool = True
    # Run the (slow) full-volume validation pass every N epochs. Full-volume
    # inference slides over an entire tomogram and is much costlier than
    # patch-level validation, so running it every epoch is wasteful. On
    # in-between epochs, checkpoint selection falls back to patch-level F1.
    # A value of 1 means "every epoch". The final epoch always runs it.
    vol_eval_every: int = 10
    optimizer: str = "adamw"
    lr: float = 1.0e-4
    betas: tuple = (0.9, 0.999)
    eps: float = 1.0e-8
    weight_decay: float = 0.0
    scheduler: str = "cosine"
    warmup_epochs: int = 3
    min_lr: float = 1.0e-6
    amp: bool = True
    grad_clip: float = 0.0
    out_dir: str = "/kaggle/working/train_results"
    save_every: int = 10
    resume: Optional[str] = None
    seed: int = 42

    @classmethod
    def from_dict(cls, d: dict) -> "TrainCfg":
        kw = {k: v for k, v in d.items() if k in cls.__dataclass_fields__}
        if "betas" in kw and isinstance(kw["betas"], list):
            kw["betas"] = tuple(kw["betas"])
        return cls(**kw)


@dataclass
class InferenceCfg:
    patch_size: int = 72
    overlap: int = 55
    pcrop: int = 25
    batch_patches: int = 4
    amp: bool = True
    write_scoremap: bool = False
    scoremap_name: str = "pyscoremap"
    segmentation_name: str = "pysegmentation"
    user_id: str = "pytorch"
    session_id: str = "0"
    out_overlay: str = "/kaggle/working/predictions"

    @classmethod
    def from_dict(cls, d: dict) -> "InferenceCfg":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


@dataclass
class LocalizeCfg:
    min_protein_size: float = 0.8
    write_copick_picks: bool = True
    write_csv: bool = False
    csv_path: str = "/kaggle/working/submission.csv"
    picks_user_id: str = "pytorch"
    picks_session_id: str = "0"

    @classmethod
    def from_dict(cls, d: dict) -> "LocalizeCfg":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


@dataclass
class Config:
    data: DataCfg = field(default_factory=DataCfg)
    model: ModelCfg = field(default_factory=ModelCfg)
    loss: LossCfg = field(default_factory=LossCfg)
    train: TrainCfg = field(default_factory=TrainCfg)
    inference: InferenceCfg = field(default_factory=InferenceCfg)
    localize: LocalizeCfg = field(default_factory=LocalizeCfg)

    # ------------------------------------------------------------------
    @classmethod
    def from_yaml(cls, path: Union[str, Path]) -> "Config":
        path = Path(path)
        with open(path, "r") as f:
            raw = yaml.safe_load(f) or {}
        return cls.from_dict(raw)

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        return cls(
            data=DataCfg.from_dict(d.get("data", {}) or {}),
            model=ModelCfg.from_dict(d.get("model", {}) or {}),
            loss=LossCfg.from_dict(d.get("loss", {}) or {}),
            train=TrainCfg.from_dict(d.get("train", {}) or {}),
            inference=InferenceCfg.from_dict(d.get("inference", {}) or {}),
            localize=LocalizeCfg.from_dict(d.get("localize", {}) or {}),
        )

    def to_dict(self) -> dict:
        return asdict(self)

    def save_yaml(self, path: Union[str, Path]) -> None:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            yaml.safe_dump(self.to_dict(), f, sort_keys=False)

    # convenience: merge CLI overrides (key path like "train.lr")
    def set(self, dotted: str, value: Any) -> "Config":
        parts = dotted.split(".")
        obj = self
        for p in parts[:-1]:
            obj = getattr(obj, p)
        # cast to the field type
        ftype = type(getattr(obj, parts[-1]))
        try:
            if ftype is tuple:
                value = tuple(value)
            else:
                value = ftype(value)
        except (TypeError, ValueError):
            pass
        setattr(obj, parts[-1], value)
        return self

### Data: copick I/O

In [ ]:
"""Copick/Zarr I/O helpers for CryoET tomograms and picks."""


import json
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import zarr


# ---------------------------------------------------------------------------
# Config blob (czii 2024 challenge layout)
# ---------------------------------------------------------------------------
CZII_CONFIG_BLOB = """{{
    "name": "czii_cryoet_mlchallenge_2024",
    "description": "2024 CZII CryoET ML Challenge training data.",
    "version": "1.0.0",
    "pickable_objects": [
        {{"name": "apo-ferritin",        "is_particle": true, "pdb_id": "4V1W", "label": 1, "color": [0,117,220,128],   "radius": 60,  "map_threshold": 0.0418}},
        {{"name": "beta-amylase",        "is_particle": true, "pdb_id": "1FA2", "label": 2, "color": [153,63,0,128],    "radius": 65,  "map_threshold": 0.035}},
        {{"name": "beta-galactosidase",  "is_particle": true, "pdb_id": "6X1Q", "label": 3, "color": [76,0,92,128],     "radius": 90,  "map_threshold": 0.0578}},
        {{"name": "ribosome",            "is_particle": true, "pdb_id": "6EK0", "label": 4, "color": [0,92,49,128],     "radius": 150, "map_threshold": 0.0374}},
        {{"name": "thyroglobulin",       "is_particle": true, "pdb_id": "6SCJ", "label": 5, "color": [43,206,72,128],   "radius": 130, "map_threshold": 0.0278}},
        {{"name": "virus-like-particle", "is_particle": true,                  "label": 6, "color": [255,204,153,128], "radius": 135, "map_threshold": 0.201}},
        {{"name": "membrane",            "is_particle": false,                 "label": 8, "color": [100,100,100,128]}},
        {{"name": "background",          "is_particle": false,                 "label": 9, "color": [10,150,200,128]}}
    ],
    "overlay_root": "{overlay_root}",
    "overlay_fs_args": {{"auto_mkdir": true}},
    "static_root": "{static_root}"
}}
"""


def write_copick_config(
    config_path: str,
    static_root: str,
    overlay_root: str,
) -> str:
    """Write a copick config JSON for the czii challenge layout."""
    Path(config_path).parent.mkdir(parents=True, exist_ok=True)
    blob = CZII_CONFIG_BLOB.format(overlay_root=overlay_root, static_root=static_root)
    with open(config_path, "w") as f:
        f.write(blob)
    return config_path


def get_copick_root(config_path: str):
    """Load a copick root from a config file (lazy import of copick)."""
    import copick

    return copick.from_file(config_path)


def list_runs(config_path: str) -> List[str]:
    root = get_copick_root(config_path)
    return [run.name for run in root.runs]


# ---------------------------------------------------------------------------
# Tomograms
# ---------------------------------------------------------------------------
def _open_zarr_volume(zarr_path: str):
    """Open a multiscale zarr group and return the level-0 array."""
    group = zarr.open(zarr_path, mode="r")
    # OME-NGFF multiscale: arrays are 0,1,2,... (0 is highest resolution)
    return group["0"]


def _fetch_tomogram(vs, tomo_algorithm: str):
    """Fetch a single tomogram from a voxel-spacing, tolerating both the
    deprecated `get_tomogram` (singular) and the new `get_tomograms` (plural,
    returns a list) copick APIs. Silences the deprecation warning either way."""
    import warnings

    tomo = None
    # New API: get_tomograms returns a list (possibly filtered by name)
    if hasattr(vs, "get_tomograms"):
        try:
            tomos = vs.get_tomograms(tomo_algorithm)
            if tomos:  # non-empty list
                tomo = tomos[0]
        except Exception:
            tomos = None
    # Fallback to deprecated singular API (suppress its DeprecationWarning)
    if tomo is None and hasattr(vs, "get_tomogram"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            tomo = vs.get_tomogram(tomo_algorithm)
    if tomo is None:
        avail = [t.tomo_type for t in vs.tomograms]
        raise ValueError(
            f"Tomogram '{tomo_algorithm}' not found. Available: {avail}"
        )
    return tomo


def get_tomogram(
    config_path: str,
    tomo_id: str,
    voxel_size: float = 10.0,
    tomo_algorithm: str = "denoised",
):
    """Return the level-0 zarr array for a tomogram (lazy, not materialized)."""
    root = get_copick_root(config_path)
    run = root.get_run(tomo_id)
    vs = run.get_voxel_spacing(voxel_size)
    if vs is None:
        raise ValueError(f"Voxel spacing {voxel_size} not found for run {tomo_id}")
    tomo = _fetch_tomogram(vs, tomo_algorithm)
    return _open_zarr_volume(tomo.zarr())


def get_tomogram_shape(config_path: str, tomo_id: str, voxel_size: float = 10.0,
                       tomo_algorithm: str = "denoised") -> Tuple[int, int, int]:
    return get_tomogram(config_path, tomo_id, voxel_size, tomo_algorithm).shape


def get_empty_target(config_path: str, tomo_id: str, voxel_size: float = 10.0,
                     tomo_algorithm: str = "denoised") -> np.ndarray:
    shape = get_tomogram_shape(config_path, tomo_id, voxel_size, tomo_algorithm)
    return np.zeros(shape, dtype=np.uint8)


# ---------------------------------------------------------------------------
# Picks (ground-truth coordinates)
# ---------------------------------------------------------------------------
def get_picks(
    config_path: str,
    tomo_id: str,
    object_name: str,
    user_id: Optional[str] = None,
    session_id: Optional[str] = None,
) -> np.ndarray:
    """Return (N,3) pick coordinates in Angstroms (x,y,z)."""
    root = get_copick_root(config_path)
    run = root.get_run(tomo_id)
    picks_list = run.get_picks(object_name, user_id=user_id, session_id=session_id)
    if not picks_list:
        return np.zeros((0, 3), dtype=np.float32)
    picks = picks_list[0]
    coords = np.array(
        [(p.location.x, p.location.y, p.location.z) for p in picks.points],
        dtype=np.float32,
    )
    return coords


def get_pickable_objects(config_path: str) -> List[dict]:
    """Return list of pickable object descriptors with name/label/radius/is_particle."""
    root = get_copick_root(config_path)
    out = []
    for obj in root.pickable_objects:
        out.append(
            {
                "name": obj.name,
                "label": obj.label,
                "radius": getattr(obj, "radius", None),
                "is_particle": obj.is_particle,
                "pdb_id": getattr(obj, "pdb_id", None),
            }
        )
    return out


def get_object_label(config_path: str, object_name: str) -> int:
    root = get_copick_root(config_path)
    return root.get_object(object_name).label


def get_object_radius_voxels(config_path: str, object_name: str,
                             voxel_size: float = 10.0) -> float:
    root = get_copick_root(config_path)
    obj = root.get_object(object_name)
    if getattr(obj, "radius", None) is None:
        return 0.0
    return float(obj.radius) / float(voxel_size)


# ---------------------------------------------------------------------------
# Segmentation read/write
# ---------------------------------------------------------------------------
def write_ome_zarr_segmentation(
    config_path: str,
    tomo_id: str,
    volume: np.ndarray,
    voxel_size: float = 10.0,
    name: str = "segmentation",
    user_id: str = "pytorch",
    session_id: str = "0",
    multilabel: bool = True,
) -> None:
    """Write a segmentation volume into the copick overlay."""
    import ome_zarr.writer

    root = get_copick_root(config_path)
    run = root.get_run(tomo_id)

    segs = run.get_segmentations(name=name, user_id=user_id, session_id=session_id)
    if len(segs) == 0 or segs[0].voxel_size != voxel_size:
        seg = run.new_segmentation(
            voxel_size=voxel_size,
            name=name,
            session_id=session_id,
            is_multilabel=multilabel,
            user_id=user_id,
        )
    else:
        seg = segs[0]

    loc = seg.zarr()
    root_group = zarr.group(loc, overwrite=True)

    axes = [
        {"name": "z", "type": "space", "unit": "angstrom"},
        {"name": "y", "type": "space", "unit": "angstrom"},
        {"name": "x", "type": "space", "unit": "angstrom"},
    ]
    transforms = [{"scale": [voxel_size, voxel_size, voxel_size], "type": "scale"}]
    ome_zarr.writer.write_multiscale(
        [volume],
        group=root_group,
        axes=axes,
        coordinate_transformations=[transforms],
        storage_options=dict(chunks=(256, 256, 256), overwrite=True),
        compute=True,
    )


def write_ome_zarr_scoremap(
    config_path: str,
    tomo_id: str,
    scoremap: np.ndarray,  # shape (C, Z, Y, X) probabilities
    voxel_size: float = 10.0,
    name: str = "pyscoremap",
    user_id: str = "pytorch",
    session_id: str = "0",
    tomo_type: str = "denoised",
) -> None:
    """Write a per-class scoremap as OME-Zarr features (channel-first)."""
    import ome_zarr.writer

    root = get_copick_root(config_path)
    run = root.get_run(tomo_id)
    tomo = _fetch_tomogram(run.get_voxel_spacing(voxel_size), tomo_type)
    feat = tomo.get_features(name)
    if feat is None:
        feat = tomo.new_features(feature_type=name)

    loc = feat.zarr()
    root_group = zarr.group(loc, overwrite=True)

    vol = np.transpose(scoremap, (3, 0, 1, 2))  # -> (X,Y,Z,C)? keep (C,Z,Y,X) per copick convention
    vol = np.ascontiguousarray(vol)

    axes = [
        {"name": "c", "type": "channel"},
        {"name": "z", "type": "space", "unit": "angstrom"},
        {"name": "y", "type": "space", "unit": "angstrom"},
        {"name": "x", "type": "space", "unit": "angstrom"},
    ]
    transforms = [
        {"scale": [voxel_size, voxel_size, voxel_size, voxel_size], "type": "scale"}
    ]
    ome_zarr.writer.write_multiscale(
        [vol],
        group=root_group,
        axes=axes,
        coordinate_transformations=[transforms],
        storage_options=dict(chunks=(1, 256, 256, 256), overwrite=True),
        compute=True,
    )


def get_segmentation(
    config_path: str,
    tomo_id: str,
    name: str = "segmentation",
    user_id: str = "pytorch",
    session_id: str = "0",
):
    """Return the level-0 zarr array for a stored segmentation."""
    root = get_copick_root(config_path)
    run = root.get_run(tomo_id)
    segs = run.get_segmentations(name=name, user_id=user_id, session_id=session_id)
    if not segs:
        raise ValueError(f"No segmentation '{name}' for {tomo_id} (user {user_id}, session {session_id})")
    return _open_zarr_volume(segs[0].zarr())

### Data: target builder (spheres)

In [ ]:
"""Label <-> model-class mapping for the CZII CryoET challenge.

copick pickable objects use *non-contiguous* integer labels:

    0  background (implicit)
    1  apo-ferritin
    2  beta-amylase
    3  beta-galactosidase
    4  ribosome
    5  thyroglobulin
    6  virus-like-particle
    8  membrane
    9  background (named object)

The model emits a contiguous softmax over `n_class` channels `0..n_class-1`.
To avoid a dead channel and an out-of-range one-hot crash, we remap copick
labels to contiguous model-class indices via ``LABEL_TO_CLASS`` and back via
``CLASS_TO_LABEL``.

With ``n_class = 8`` (the default), the mapping is:

    copick label 0 (background)      -> class 0
    copick label 1 (apo-ferritin)    -> class 1
    copick label 2 (beta-amylase)    -> class 2
    copick label 3 (beta-galactos.)  -> class 3
    copick label 4 (ribosome)        -> class 4
    copick label 5 (thyroglobulin)   -> class 5
    copick label 6 (virus-like-part) -> class 6
    copick label 8 (membrane)        -> class 7

If membrane is not used, class 7 has no target signal; ``ACTIVE_CLASSES``
lists the classes that actually appear in targets and should participate in
loss weighting / metric aggregation / model selection.
"""
from __future__ import annotations


# --- Namespace shim: expose flat notebook globals as module-like objects so the
# --- backend-style module bodies below (which call copick_io.X / split_utils.X)
# --- run unchanged in this notebook's flat global namespace.
import types as _types
_copick_io_names = [
    "write_copick_config", "get_copick_root", "list_runs",
    "_open_zarr_volume", "_fetch_tomogram", "get_tomogram", "get_tomogram_shape",
    "get_empty_target", "get_picks", "get_pickable_objects",
    "get_object_label", "get_object_radius_voxels",
    "write_ome_zarr_segmentation", "write_ome_zarr_scoremap", "get_segmentation",
]
class _LazyCopickIO:
    def __getattr__(self, name):
        if name.startswith("__") and name.endswith("__"):
            raise AttributeError(name)
        try:
            return globals()[name]
        except KeyError:
            raise AttributeError(f"copick_io.{name} not defined in notebook globals")
copick_io = _LazyCopickIO()
class _LazySplitUtils:
    @property
    def split_runs(self):
        return globals()["split_runs"]
split_utils = _LazySplitUtils()
# Augment3D is referenced by dataset.py as a bare global (class), already defined above.


from typing import Dict, List

# copick label -> contiguous model class index
LABEL_TO_CLASS: Dict[int, int] = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,
    8: 7,
}

# contiguous model class index -> copick label
CLASS_TO_LABEL: Dict[int, int] = {v: k for k, v in LABEL_TO_CLASS.items()}

# Particle (foreground) copick labels, in scored order.
PARTICLE_LABELS: List[int] = [1, 2, 3, 4, 5, 6]

# Particle (foreground) model class indices.
PARTICLE_CLASSES: List[int] = [LABEL_TO_CLASS[l] for l in PARTICLE_LABELS]

# Classes that actually carry target signal when membrane is NOT stamped.
# (background + the six particles). Class 7 (membrane) is excluded until
# membrane targets are explicitly built.
ACTIVE_CLASSES: List[int] = [0] + PARTICLE_CLASSES

# Name table for logging / localization.
CLASS_TO_NAME: Dict[int, str] = {
    0: "background",
    1: "apo-ferritin",
    2: "beta-amylase",
    3: "beta-galactosidase",
    4: "ribosome",
    5: "thyroglobulin",
    6: "virus-like-particle",
    7: "membrane",
}


def remap_volume(vol, mapping: Dict[int, int] = None) -> "vol":
    """Remap an integer label volume in-place-safe using a lookup table.

    Values not present in ``mapping`` are mapped to 0 (treated as background).
    Works on numpy uint8 arrays and torch integer tensors.
    """
    import numpy as np

    m = mapping or LABEL_TO_CLASS
    src_dtype = vol.dtype if hasattr(vol, "dtype") else None
    # build a LUT large enough to cover any value present in the input
    max_key = max(max(m.keys()), int(vol.max()) if hasattr(vol, "max") else 0)
    table = np.zeros(max_key + 1, dtype=np.int64)
    for k, v in m.items():
        if k <= max_key:
            table[k] = v
    if isinstance(vol, np.ndarray):
        return table[vol.astype(np.int64)].astype(src_dtype)
    # torch tensor
    import torch

    t = torch.from_numpy(table).to(vol.device).to(vol.dtype)
    mask = (vol >= 0) & (vol <= max_key)
    out = torch.zeros_like(vol)
    out[mask] = t[vol[mask].long()]
    return out


def inverse_remap_volume(vol) -> "vol":
    """Map model class indices back to copick labels."""
    import numpy as np

    inv = {v: k for k, v in LABEL_TO_CLASS.items()}
    return remap_volume(vol, inv)


"""Build sphere-based segmentation targets from copick picks (port of DeepFindET).

Targets are written with *contiguous model-class indices* (see ``labels.py``),
so copick label 8 (membrane) maps to class 7, and particle labels 1..6 map to
classes 1..6. Background is 0. This keeps the dense label volume inside
``0..n_class-1`` so downstream one-hot / softmax / argmax are well-defined.
"""



import numpy as np
from scipy import ndimage
from tqdm import tqdm



def make_sphere(diameter: int, radius: float) -> np.ndarray:
    """Return a cubic uint8 mask of a filled sphere."""
    d = int(diameter)
    if d <= 0:
        return np.zeros((0, 0, 0), dtype=np.uint8)
    z = np.arange(d) - d // 2
    y = np.arange(d) - d // 2
    x = np.arange(d) - d // 2
    zz, yy, xx = np.meshgrid(z, y, x, indexing="ij")
    dist = np.sqrt(zz ** 2 + yy ** 2 + xx ** 2)
    return (dist <= radius).astype(np.uint8)


def stamp_sphere(
    target: np.ndarray, cx: int, cy: int, cz: int, sphere: np.ndarray, label: int
) -> None:
    """Stamp a sphere mask (in-place) into target at integer center (x,y,z)."""
    if sphere.size == 0:
        return
    d = sphere.shape[0]
    half = d // 2

    x0, x1 = cx - half, cx - half + d
    y0, y1 = cy - half, cy - half + d
    z0, z1 = cz - half, cz - half + d

    # bounds inside target (Z,Y,X)
    tz0, tz1 = max(z0, 0), min(z1, target.shape[0])
    ty0, ty1 = max(y0, 0), min(y1, target.shape[1])
    tx0, tx1 = max(x0, 0), min(x1, target.shape[2])

    if tz0 >= tz1 or ty0 >= ty1 or tx0 >= tx1:
        return

    # corresponding region in sphere
    sz0, sz1 = tz0 - z0, tz1 - z0
    sy0, sy1 = ty0 - y0, ty1 - y0
    sx0, sx1 = tx0 - x0, tx1 - x0

    region = target[tz0:tz1, ty0:ty1, tx0:tx1]
    mask = sphere[sz0:sz1, sy0:sy1, sx0:sx1].astype(bool)
    np.maximum(region, label * mask.astype(region.dtype), out=region, where=mask)


def build_targets_for_run(
    config_path: str,
    tomo_id: str,
    voxel_size: float = 10.0,
    particle_targets: dict | None = None,
    seg_targets: dict | None = None,
) -> np.ndarray:
    """Build a uint8 segmentation target volume for a single run.

    Args:
        config_path: copick config path.
        tomo_id: run name.
        voxel_size: voxel size in Angstrom.
        particle_targets: dict {object_name: {"label":..,"user_id":..,"session_id":..,"radius":..}}.
            If None, all particle pickable objects are used.
        seg_targets: dict {seg_name: {"label":..,"user_id":..,"session_id":..}} for membrane-like
            pre-existing segmentations to overlay.
    """
    import zarr

    root = copick_io.get_copick_root(config_path)

    # Determine target objects
    if particle_targets is None:
        particle_targets = {}
        for obj in root.pickable_objects:
            if obj.is_particle:
                r = getattr(obj, "radius", None)
                # remap copick label -> contiguous model-class index
                particle_targets[obj.name] = {
                    "label": LABEL_TO_CLASS.get(obj.label, 0),
                    "user_id": None,
                    "session_id": None,
                    "radius": (r / voxel_size) if r else 0.0,
                }

    target_vol = copick_io.get_empty_target(config_path, tomo_id, voxel_size)

    # Overlay existing segmentations (e.g. membrane) if provided.
    # `info["label"]` here is expected to be a *model-class index* (already
    # remapped by the caller); we additionally remap the stored volume in case
    # it still carries raw copick labels.
    if seg_targets:
        for name, info in seg_targets.items():
            segs = root.get_run(tomo_id).get_segmentations(
                name=name,
                user_id=info.get("user_id"),
                session_id=info.get("session_id"),
                voxel_size=voxel_size,
                is_multilabel=False,
            )
            for seg in segs:
                vol = zarr.open(seg.zarr(), mode="r")["0"][:]
                vol = remap_volume(vol.astype(np.uint8))
                np.maximum(target_vol, vol * info["label"], out=target_vol)

    # Precompute sphere masks per class (radius in voxels)
    spheres = {}
    for name, info in particle_targets.items():
        r = info["radius"]
        if r <= 0:
            continue
        diameter = int(np.ceil(2 * r)) + 2
        spheres[info["label"]] = make_sphere(diameter, r)

    # Stamp particles
    for name, info in particle_targets.items():
        label = info["label"]
        sphere = spheres.get(label)
        if sphere is None or sphere.size == 0:
            continue
        coords = copick_io.get_picks(
            config_path, tomo_id, name,
            user_id=info.get("user_id"),
            session_id=info.get("session_id"),
        )
        if coords.shape[0] == 0:
            continue
        # Angstrom -> voxel
        vcoords = coords / voxel_size
        for (xa, ya, za) in vcoords:
            stamp_sphere(target_vol, int(round(xa)), int(round(ya)), int(round(za)),
                         sphere, label)
    return target_vol


def build_targets(
    config_path: str,
    tomo_ids: list[str] | None = None,
    voxel_size: float = 10.0,
    out_name: str = "pytargets",
    out_user_id: str = "pytorch",
    out_session_id: str = "0",
    particle_targets: dict | None = None,
    seg_targets: dict | None = None,
) -> None:
    """Build and write segmentation targets for all (or given) runs."""
    root = copick_io.get_copick_root(config_path)
    if tomo_ids is None:
        tomo_ids = [run.name for run in root.runs]

    if particle_targets is None:
        particle_targets = {}
        for obj in root.pickable_objects:
            if obj.is_particle:
                r = getattr(obj, "radius", None)
                particle_targets[obj.name] = {
                    "label": LABEL_TO_CLASS.get(obj.label, 0),
                    "user_id": None,
                    "session_id": None,
                    "radius": (r / voxel_size) if r else 0.0,
                }

    for tomo_id in tqdm(tomo_ids, desc="Building targets"):
        target = build_targets_for_run(
            config_path, tomo_id, voxel_size, particle_targets, seg_targets
        )
        copick_io.write_ome_zarr_segmentation(
            config_path, tomo_id, target, voxel_size,
            name=out_name, user_id=out_user_id, session_id=out_session_id,
            multilabel=True,
        )



### Data: 3D augmentation

In [ ]:
"""3D data augmentations operating on torch tensors (image + integer label)."""


import random
from typing import Tuple

import numpy as np
import torch
import torch.nn.functional as F


class Augment3D:
    """Apply a random subset of 3D augmentations to (image, label) patches.

    image: torch.FloatTensor of shape (1, D, D, D)
    label: torch.LongTensor  of shape (D, D, D) with integer class indices
    """

    def __init__(
        self,
        p_flip: float = 0.5,
        p_rot180: float = 0.5,
        p_noise: float = 0.3,
        p_blur: float = 0.3,
        p_brightness: float = 0.4,
        p_contrast: float = 0.4,
        p_intensity: float = 0.4,
        noise_std: float = 0.05,
        blur_sigma: Tuple[float, float] = (0.5, 1.25),
    ):
        self.p_flip = p_flip
        self.p_rot180 = p_rot180
        self.p_noise = p_noise
        self.p_blur = p_blur
        self.p_brightness = p_brightness
        self.p_contrast = p_contrast
        self.p_intensity = p_intensity
        self.noise_std = noise_std
        self.blur_sigma = blur_sigma

    def __call__(
        self, image: torch.Tensor, label: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Random flips along any of the 3 spatial axes (cheap, label-safe)
        # image: (1,1,D,D,D) -> spatial dims 2,3,4 ; label: (D,D,D) -> 0,1,2
        for ax in (0, 1, 2):
            if random.random() < self.p_flip:
                image = torch.flip(image, dims=[ax + 2])
                label = torch.flip(label, dims=[ax])

        # 180-degree rotation around a random pair of spatial axes (label-safe)
        if random.random() < self.p_rot180:
            pair = random.choice([(0, 1), (0, 2), (1, 2)])
            label = torch.rot90(label, k=2, dims=list(pair))
            image = torch.rot90(image, k=2, dims=[d + 2 for d in pair])

        # Intensity augmentations (image only)
        if random.random() < self.p_brightness:
            image = image + float(np.random.uniform(-0.3, 0.3))
        if random.random() < self.p_intensity:
            image = image * float(np.random.uniform(0.8, 1.2))
        if random.random() < self.p_contrast:
            mean = image.mean()
            image = mean + float(np.random.uniform(0.7, 1.3)) * (image - mean)
        if random.random() < self.p_noise:
            std = float(np.random.uniform(0.0, self.noise_std))
            image = image + torch.randn_like(image) * std
        if random.random() < self.p_blur:
            sigma = float(np.random.uniform(*self.blur_sigma))
            image = _gaussian_blur_3d(image, sigma)

        return image, label


def _gaussian_blur_3d(x: torch.Tensor, sigma: float) -> torch.Tensor:
    """Separable 3D gaussian blur on (1,1,D,D,D)."""
    if sigma <= 0:
        return x
    radius = max(1, int(3 * sigma))
    ax = torch.arange(-radius, radius + 1, dtype=x.dtype, device=x.device)
    k1d = torch.exp(-(ax ** 2) / (2 * sigma ** 2))
    k1d = k1d / k1d.sum()
    # reshape to conv kernels (1,1,K,1,1),(1,1,1,K,1),(1,1,1,1,K)
    pad = radius
    k = k1d.view(1, 1, -1, 1, 1)
    x = F.conv3d(x, k, padding=(pad, 0, 0))
    k = k1d.view(1, 1, 1, -1, 1)
    x = F.conv3d(x, k, padding=(0, pad, 0))
    k = k1d.view(1, 1, 1, 1, -1)
    x = F.conv3d(x, k, padding=(0, 0, pad))
    return x

### Data: splits

In [ ]:
"""Dataset splitting helpers."""


import random
from typing import List, Tuple


def split_runs(
    tomo_ids: List[str],
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> Tuple[List[str], List[str], List[str]]:
    n = len(tomo_ids)
    rng = random.Random(seed)
    ids = list(tomo_ids)
    rng.shuffle(ids)
    n_train = int(round(n * train_ratio))
    n_val = int(round(n * val_ratio))
    train = ids[:n_train]
    val = ids[n_train : n_train + n_val]
    test = ids[n_train + n_val :]
    return train, val, test

### Data: dataset (class-balanced bootstrap patch sampler)

In [ ]:
"""PyTorch Datasets for CryoET 3D patch sampling (class-balanced bootstrap)."""
from __future__ import annotations



import random
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from torch.utils.data import IterableDataset



# ---------------------------------------------------------------------------
# Pick indexing across runs
# ---------------------------------------------------------------------------
def index_picks(
    config_path: str,
    tomo_ids: List[str],
    targets: Optional[Dict[str, dict]] = None,
    voxel_size: float = 10.0,
) -> List[Tuple[str, float, float, float, int]]:
    """Flatten all picks across runs into a list of (tomo_id, x, y, z, class).

    Coordinates are in VOXEL units (Angstrom / voxel_size). The returned class
    index is the *contiguous model-class index* (copick label remapped via
    ``LABEL_TO_CLASS``), so it is always in ``0..n_class-1``.
    """
    root = copick_io.get_copick_root(config_path)
    if targets is None:
        targets = {}
        for obj in root.pickable_objects:
            if obj.is_particle:
                targets[obj.name] = {"user_id": None, "session_id": None}

    flat = []
    for tomo_id in tomo_ids:
        for name, info in targets.items():
            obj = root.get_object(name)
            cls = LABEL_TO_CLASS.get(obj.label, 0)  # model-class index
            coords = copick_io.get_picks(
                config_path, tomo_id, name,
                user_id=info.get("user_id"),
                session_id=info.get("session_id"),
            )
            if coords.shape[0] == 0:
                continue
            vcoords = coords / voxel_size  # (N,3) x,y,z in voxels
            for (x, y, z) in vcoords:
                flat.append((tomo_id, float(x), float(y), float(z), int(cls)))
    return flat


def get_class_counts(picks: List[Tuple[str, float, float, float, int]]) -> Dict[int, int]:
    counts: Dict[int, int] = defaultdict(int)
    for *_p, label in picks:
        counts[label] += 1
    return dict(counts)


# ---------------------------------------------------------------------------
# Patch position sampler
# ---------------------------------------------------------------------------
def patch_position(
    tomo_shape: Tuple[int, int, int],
    p_in: int,
    l_rnd: int,
    cx: float,
    cy: float,
    cz: float,
) -> Tuple[int, int, int]:
    """Compute integer (x, y, z) patch center with random shift, clamped to bounds."""
    x = int(cx) + np.random.randint(-l_rnd, l_rnd + 1)
    y = int(cy) + np.random.randint(-l_rnd, l_rnd + 1)
    z = int(cz) + np.random.randint(-l_rnd, l_rnd + 1)
    # tomo_shape is (Z, Y, X)
    x = min(max(x, p_in), tomo_shape[2] - p_in)
    y = min(max(y, p_in), tomo_shape[1] - p_in)
    z = min(max(z, p_in), tomo_shape[0] - p_in)
    return x, y, z


# ---------------------------------------------------------------------------
# Tomogram pool (in-memory subset, swappable)
# ---------------------------------------------------------------------------
class TomogramPool:
    """Holds a subset of tomograms + targets resident in RAM."""

    def __init__(
        self,
        config_path: str,
        target_name: str,
        target_user_id: str,
        target_session_id: str,
        voxel_size: float = 10.0,
        tomo_algorithm: str = "denoised",
    ):
        self.config_path = config_path
        self.target_name = target_name
        self.target_user_id = target_user_id
        self.target_session_id = target_session_id
        self.voxel_size = voxel_size
        self.tomo_algorithm = tomo_algorithm
        self.data: Dict[str, np.ndarray] = {}   # tomo_id -> denoised vol (np.float32)
        self.targets: Dict[str, np.ndarray] = {}  # tomo_id -> uint8 labels

    def load(self, tomo_ids: List[str]) -> None:
        self.data.clear()
        self.targets.clear()
        for tid in tomo_ids:
            tomo = copick_io.get_tomogram(
                self.config_path, tid, self.voxel_size, self.tomo_algorithm
            )[:]
            self.data[tid] = tomo.astype(np.float32)
            seg = copick_io.get_segmentation(
                self.config_path, tid,
                name=self.target_name,
                user_id=self.target_user_id,
                session_id=self.target_session_id,
            )[:]
            self.targets[tid] = seg.astype(np.uint8)

    def has(self, tomo_id: str) -> bool:
        return tomo_id in self.data

    def shape(self, tomo_id: str) -> Tuple[int, int, int]:
        return self.data[tomo_id].shape

    def get_patch(
        self, tomo_id: str, x: int, y: int, z: int, p_in: int
    ) -> Tuple[np.ndarray, np.ndarray]:
        p = p_in
        data = self.data[tomo_id][z - p:z + p, y - p:y + p, x - p:x + p]
        tgt = self.targets[tomo_id][z - p:z + p, y - p:y + p, x - p:x + p]
        return data.copy(), tgt.copy()


# ---------------------------------------------------------------------------
# Iterable dataset
# ---------------------------------------------------------------------------
class CryoETPatchDataset(IterableDataset):
    """Yields a fixed number of (image, label) patch batches per epoch.

    - Class-balanced bootstrap: each step picks `batch_size` picks, balanced
      across foreground classes, with a fraction of pure-background patches.
    - Tomogram pool swaps every `n_sub_epoch` (handled externally by calling
      `set_pool`); this class just samples from the current pool.
    """

    def __init__(
        self,
        config_path: str,
        tomo_ids: List[str],
        dim_in: int,
        batch_size: int,
        steps: int,
        l_rnd: int = 15,
        background_ratio: float = 0.30,
        voxel_size: float = 10.0,
        tomo_algorithm: str = "denoised",
        target_name: str = "pytargets",
        target_user_id: str = "pytorch",
        target_session_id: str = "0",
        targets: Optional[Dict[str, dict]] = None,
        augment: Optional[Augment3D] = None,
        pool: Optional[TomogramPool] = None,
        n_sub_epoch: int = 10,
        sample_size: int = 5,
        seed: int = 42,
    ):
        super().__init__()
        self.config_path = config_path
        self.tomo_ids = list(tomo_ids)
        self.dim_in = dim_in
        self.p_in = dim_in // 2
        self.batch_size = batch_size
        self.steps = steps
        self.l_rnd = l_rnd
        self.background_ratio = background_ratio
        self.voxel_size = voxel_size
        self.tomo_algorithm = tomo_algorithm
        self.target_name = target_name
        self.target_user_id = target_user_id
        self.target_session_id = target_session_id
        self.targets = targets
        self.augment = augment
        self.n_sub_epoch = n_sub_epoch
        self.sample_size = sample_size
        self.seed = seed
        self._rng = random.Random(seed)

        # Index picks for all provided tomo_ids (class indices already remapped)
        self.picks = index_picks(config_path, tomo_ids, targets, voxel_size)
        # group by class index
        self.by_class: Dict[int, List[int]] = defaultdict(list)
        for i, (*_p, label) in enumerate(self.picks):
            self.by_class[label].append(i)
        self.class_labels = sorted(self.by_class.keys())
        if not self.class_labels:
            raise ValueError("No picks found for the given tomo_ids/targets.")

        # Per-tomogram pick coordinate arrays (voxel units) for background
        # rejection. Keyed by tomo_id -> (N,3) float array of (x,y,z).
        self._picks_by_tomo: Dict[str, np.ndarray] = defaultdict(list)
        for (t, x, y, z, _cls) in self.picks:
            self._picks_by_tomo[t].append((x, y, z))
        self._picks_by_tomo = {
            t: np.asarray(v, dtype=np.float32) for t, v in self._picks_by_tomo.items()
        }
        # Max particle radius in voxels (used as exclusion distance for bg
        # sampling). Default 15 voxels (150 A / 10 A) if unknown.
        self._bg_exclude = max(15.0, self._max_radius_voxels(config_path, voxel_size))

        # Tomogram pool
        self.pool = pool or TomogramPool(
            config_path, target_name, target_user_id, target_session_id,
            voxel_size, tomo_algorithm,
        )
        self._pool_loaded_ids: List[str] = []
        self._swap_counter = 0

    # -- pool management -------------------------------------------------
    def _ensure_pool(self) -> None:
        """Load a fresh random subset of tomograms if pool empty/stale."""
        needed = set(t for (t, *_p) in self.picks)
        if self._pool_loaded_ids and self._swap_counter < self.n_sub_epoch:
            # still in current subset window; ensure needed tomos loaded
            missing = [t for t in self._pool_loaded_ids if not self.pool.has(t)]
            if not missing:
                return
        # pick a new random subset that has picks
        candidates = [t for t in self.tomo_ids if t in needed]
        if not candidates:
            candidates = list(self.tomo_ids)
        k = min(self.sample_size, len(candidates))
        subset = self._rng.sample(candidates, k)
        self.pool.load(subset)
        self._pool_loaded_ids = subset
        self._swap_counter = 0

    def set_pool(self, pool: TomogramPool) -> None:
        self.pool = pool

    # -- helpers ---------------------------------------------------------
    @staticmethod
    def _max_radius_voxels(config_path: str, voxel_size: float) -> float:
        """Largest particle radius in voxels across pickable objects."""
        try:
            objs = copick_io.get_pickable_objects(config_path)
        except Exception:
            return 15.0
        r = 0.0
        for o in objs:
            if o.get("is_particle") and o.get("radius"):
                r = max(r, float(o["radius"]) / float(voxel_size))
        return r or 15.0

    # -- sampling --------------------------------------------------------
    def _sample_indices(self) -> List[int]:
        n_bg = int(round(self.batch_size * self.background_ratio))
        n_fg = self.batch_size - n_bg
        chosen: List[int] = []
        if n_fg > 0:
            # cycle through foreground particle classes for balance
            fg_labels = [l for l in self.class_labels if l in PARTICLE_CLASSES]
            if not fg_labels:
                fg_labels = list(self.class_labels)
            for i in range(n_fg):
                lbl = fg_labels[i % len(fg_labels)]
                pool = self.by_class[lbl]
                chosen.append(self._rng.choice(pool))
        return chosen

    def _sample_background_index(self) -> Tuple[str, float, float, float]:
        """Random background location inside a loaded tomo, avoiding picks.

        Rejects candidates within ``_bg_exclude`` voxels (plus ``l_rnd`` jitter)
        of any particle pick so that "background" patches actually contain
        background rather than accidentally overlapping a particle.
        """
        tid = self._rng.choice(self._pool_loaded_ids)
        shape = self.pool.shape(tid)
        p = self.p_in
        picks = self._picks_by_tomo.get(tid)
        excl = self._bg_exclude + self.l_rnd
        for _ in range(16):  # try up to 16 times to find a clean location
            x = self._rng.randint(p, shape[2] - p)
            y = self._rng.randint(p, shape[1] - p)
            z = self._rng.randint(p, shape[0] - p)
            if picks is None or picks.shape[0] == 0:
                return tid, float(x), float(y), float(z)
            d = np.sqrt(((picks - np.array([x, y, z], dtype=np.float32)) ** 2).sum(1))
            if d.min() > excl:
                return tid, float(x), float(y), float(z)
        # fall back to the last candidate if we couldn't find a clean one
        return tid, float(x), float(y), float(z)

    # -- iteration -------------------------------------------------------
    def __iter__(self):
        self._ensure_pool()
        worker = torch.utils.data.get_worker_info()
        worker_id = worker.id if worker is not None else 0
        num_workers = worker.num_workers if worker is not None else 1
        rng = random.Random(self.seed + worker_id)
        steps_for_worker = self.steps // num_workers + (1 if worker_id < self.steps % num_workers else 0)

        for _ in range(steps_for_worker):
            self._swap_counter += 1
            if self._swap_counter >= self.n_sub_epoch:
                self._ensure_pool()

            imgs = []
            lbls = []
            fg_idxs = self._sample_indices()
            n_bg = self.batch_size - len(fg_idxs)

            for idx in fg_idxs:
                tid, cx, cy, cz, label = self.picks[idx]
                if not self.pool.has(tid):
                    # fallback: load this tomo if pool missing it
                    self.pool.load([tid])
                    self._pool_loaded_ids = list(set(self._pool_loaded_ids + [tid]))
                shape = self.pool.shape(tid)
                x, y, z = patch_position(shape, self.p_in, self.l_rnd, cx, cy, cz)
                data, tgt = self.pool.get_patch(tid, x, y, z, self.p_in)
                # normalize per-patch
                data = (data - data.mean()) / (data.std() + 1e-8)
                imgs.append(data)
                lbls.append(tgt)

            # background patches
            for _ in range(n_bg):
                tid, cx, cy, cz = self._sample_background_index()
                shape = self.pool.shape(tid)
                x, y, z = patch_position(shape, self.p_in, self.l_rnd, cx, cy, cz)
                data, tgt = self.pool.get_patch(tid, x, y, z, self.p_in)
                data = (data - data.mean()) / (data.std() + 1e-8)
                imgs.append(data)
                lbls.append(tgt)

            # shuffle batch so background isn't always last
            order = list(range(len(imgs)))
            rng.shuffle(order)
            imgs = [imgs[i] for i in order]
            lbls = [lbls[i] for i in order]

            img_t = torch.from_numpy(np.stack(imgs)).float().unsqueeze(1)  # (B,1,D,D,D)
            lbl_t = torch.from_numpy(np.stack(lbls)).long()                 # (B,D,D,D)

            if self.augment is not None:
                img_t, lbl_t = self.augment(img_t, lbl_t)

            yield img_t, lbl_t



### Models: blocks + U-Net / ResU-Net / Attention U-Net + factory

In [ ]:
"""3D building blocks: conv blocks, residual blocks, attention blocks.

Normalization uses GroupNorm (not BatchNorm) so training is stable at the small
per-GPU batches typical of 3D segmentation (e.g. 4/GPU on 2x T4 with DataParallel).
GroupNorm is independent of batch size and avoids running-stat drift between
train and eval.
"""
from __future__ import annotations



import torch
import torch.nn as nn


def _norm(channels: int, groups: int = 8) -> nn.Module:
    """GroupNorm with at most `groups` groups (clamped to channel count)."""
    g = min(groups, channels)
    # ensure groups divides channels; if not, fall back to a divisor
    while channels % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, channels)


class ConvBlock(nn.Module):
    """Two 3x3x3 convs with GroupNorm + LeakyReLU."""

    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            _norm(out_ch),
            nn.LeakyReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            _norm(out_ch),
            nn.LeakyReLU(inplace=True),
        ]
        if dropout and dropout > 0:
            layers.append(nn.Dropout3d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    """Residual conv block with 1x1x1 shortcut when channels differ.

    Uses GroupNorm instead of BatchNorm for stability at small per-GPU batches.
    """

    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn1 = _norm(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = _norm(out_ch)
        self.act = nn.LeakyReLU(inplace=True)
        self.shortcut = (
            nn.Conv3d(in_ch, out_ch, 1, bias=False)
            if in_ch != out_ch
            else nn.Identity()
        )
        self.drop = nn.Dropout3d(dropout) if dropout and dropout > 0 else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.drop(out)
        return self.act(out + identity)


class AttentionBlock(nn.Module):
    """Gating attention block for attention U-Net."""

    def __init__(self, gate_ch: int, skip_ch: int, inter_ch: int):
        super().__init__()
        self.theta = nn.Conv3d(skip_ch, inter_ch, 1, bias=False)
        self.phi = nn.Conv3d(gate_ch, inter_ch, 1, bias=False)
        self.psi = nn.Conv3d(inter_ch, 1, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.gate = nn.Sigmoid()
        self.upsample = nn.Upsample(scale_factor=2, mode="trilinear", align_corners=False)

    def forward(self, x, g):
        theta = self.theta(x)
        phi = self.phi(g)
        # upsample gate to match skip spatial size
        phi = self.upsample(phi)
        if phi.shape[2:] != theta.shape[2:]:
            phi = nn.functional.interpolate(
                phi, size=theta.shape[2:], mode="trilinear", align_corners=False
            )
        h = self.act(theta + phi)
        psi = self.gate(self.psi(h))
        # broadcast psi to skip channels
        psi = psi.expand(-1, x.shape[1], -1, -1, -1)
        return x * psi


"""Plain 3D U-Net."""


import torch
import torch.nn as nn



class UNet3D(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        n_class: int = 8,
        filters=(32, 48, 64),
        dropout: float = 0.0,
    ):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pools = nn.ModuleList()
        chs = [in_channels] + list(filters[:-1])
        for i, f in enumerate(filters[:-1]):
            self.downs.append(ConvBlock(chs[i], f, dropout))
            self.pools.append(nn.MaxPool3d(2))
        # bottleneck
        bot_f = filters[-1]
        self.bottleneck = nn.Sequential(
            ConvBlock(chs[-1], bot_f, dropout),
            ConvBlock(bot_f, bot_f, dropout),
        )
        # decoder
        rev = list(reversed(filters[:-1]))
        up_chs = [bot_f] + list(rev[:-1])
        for i, f in enumerate(rev):
            self.ups.append(nn.ConvTranspose3d(up_chs[i], f, 2, stride=2))
            self.ups.append(ConvBlock(f * 2, f, dropout))
        self.head = nn.Conv3d(f, n_class, 1)

    def forward(self, x):
        skips = []
        for down, pool in zip(self.downs, self.pools):
            x = down(x)
            skips.append(x)
            x = pool(x)
        x = self.bottleneck(x)
        skips = list(reversed(skips))
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(
                    x, size=skip.shape[2:], mode="trilinear", align_corners=False
                )
            x = torch.cat([x, skip], dim=1)
            x = self.ups[i + 1](x)
        return self.head(x)


"""Residual 3D U-Net (primary model)."""


import torch
import torch.nn as nn



class ResUNet3D(nn.Module):
    """3D residual U-Net with two pooling stages (dim_in must be multiple of 4)."""

    def __init__(
        self,
        in_channels: int = 1,
        n_class: int = 8,
        filters=(48, 64, 128),
        dropout: float = 0.0,
    ):
        super().__init__()
        self.downs = nn.ModuleList()
        self.pools = nn.ModuleList()
        chs = [in_channels] + list(filters[:-1])
        for i, f in enumerate(filters[:-1]):
            self.downs.append(ResidualBlock(chs[i], f, dropout))
            self.pools.append(nn.MaxPool3d(2))

        bot_f = filters[-1]
        self.bottleneck = nn.Sequential(
            ResidualBlock(chs[-1], bot_f, dropout),
            ResidualBlock(bot_f, bot_f, dropout),
            ResidualBlock(bot_f, bot_f, dropout),
            ResidualBlock(bot_f, bot_f, dropout),
        )

        self.ups = nn.ModuleList()
        rev = list(reversed(filters[:-1]))
        up_chs = [bot_f] + list(rev[:-1])
        for i, f in enumerate(rev):
            self.ups.append(nn.ConvTranspose3d(up_chs[i], f, 2, stride=2))
            self.ups.append(ResidualBlock(f * 2, f, dropout))
            self.ups.append(ResidualBlock(f, f, dropout))
        self.head = nn.Conv3d(f, n_class, 1)

    def forward(self, x):
        skips = []
        for down, pool in zip(self.downs, self.pools):
            x = down(x)
            skips.append(x)
            x = pool(x)
        x = self.bottleneck(x)
        skips = list(reversed(skips))
        idx = 0
        for skip in skips:
            x = self.ups[idx](x)  # upsample
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(
                    x, size=skip.shape[2:], mode="trilinear", align_corners=False
                )
            x = torch.cat([x, skip], dim=1)
            x = self.ups[idx + 1](x)
            x = self.ups[idx + 2](x)
            idx += 3
        return self.head(x)


"""Attention 3D U-Net."""


import torch
import torch.nn as nn



class AttentionUNet3D(nn.Module):
    def __init__(
        self,
        in_channels: int = 1,
        n_class: int = 8,
        filters=(32, 48, 64),
        dropout: float = 0.2,
    ):
        super().__init__()
        self.downs = nn.ModuleList()
        self.pools = nn.ModuleList()
        chs = [in_channels] + list(filters[:-1])
        for i, f in enumerate(filters[:-1]):
            self.downs.append(ConvBlock(chs[i], f, dropout))
            self.pools.append(nn.MaxPool3d(2))

        bot_f = filters[-1]
        self.bottleneck = nn.Sequential(
            ConvBlock(chs[-1], bot_f, dropout),
            ConvBlock(bot_f, bot_f, dropout),
        )

        self.ups = nn.ModuleList()
        self.atts = nn.ModuleList()
        rev = list(reversed(filters[:-1]))
        up_chs = [bot_f] + list(rev[:-1])
        for i, f in enumerate(rev):
            self.ups.append(nn.ConvTranspose3d(up_chs[i], f, 2, stride=2))
            self.atts.append(AttentionBlock(gate_ch=up_chs[i], skip_ch=f, inter_ch=f))
            self.ups.append(ConvBlock(f * 2, f, dropout))
        self.head = nn.Conv3d(f, n_class, 1)

    def forward(self, x):
        skips = []
        for down, pool in zip(self.downs, self.pools):
            x = down(x)
            skips.append(x)
            x = pool(x)
        x = self.bottleneck(x)
        skips = list(reversed(skips))
        idx = 0
        for skip in skips:
            upconv = self.ups[idx](x)
            if upconv.shape[2:] != skip.shape[2:]:
                upconv = nn.functional.interpolate(
                    upconv, size=skip.shape[2:], mode="trilinear", align_corners=False
                )
            att = self.atts[idx // 2](skip, x)
            x = torch.cat([upconv, att], dim=1)
            x = self.ups[idx + 1](x)
            idx += 2
        return self.head(x)


"""Model factory."""


import torch.nn as nn



_REGISTRY = {
    "unet": UNet3D,
    "res_unet": ResUNet3D,
    "attention_unet": AttentionUNet3D,
}


def build_model(
    name: str,
    in_channels: int = 1,
    n_class: int = 8,
    filters=(48, 64, 128),
    dropout: float = 0.0,
) -> nn.Module:
    if name not in _REGISTRY:
        raise ValueError(f"Unknown model '{name}'. Choices: {list(_REGISTRY)}")
    cls = _REGISTRY[name]
    return cls(in_channels=in_channels, n_class=n_class, filters=tuple(filters), dropout=dropout)



### Losses (Tversky, Focal-Tversky, Dice, CE+Tversky)

In [ ]:
"""Loss functions for 3D multi-class segmentation (PyTorch, logits + integer targets).

Targets are expected to carry *contiguous model-class indices* in
``0..n_class-1`` (see ``labels.py``). Inactive classes (e.g. membrane when not
stamped) receive zero weight in the CE term and are excluded from the Tversky
sum so they don't inflate the loss or waste gradient.
"""
from __future__ import annotations



from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


def _softmax_probs(logits: torch.Tensor) -> torch.Tensor:
    return F.softmax(logits, dim=1)


def _resolve_class_weights(
    class_weights, n_class: int, active_classes: Optional[List[int]]
) -> Optional[torch.Tensor]:
    """Combine explicit per-class weights with an active-class mask.

    Any class not in ``active_classes`` is forced to zero weight so inactive
    heads (e.g. membrane when not trained) don't pull the model.
    """
    if class_weights is None:
        if active_classes is None:
            return None
        w = torch.zeros(n_class, dtype=torch.float32)
        for c in active_classes:
            w[c] = 1.0
        return w
    w = torch.as_tensor(class_weights, dtype=torch.float32)
    if w.numel() != n_class:
        # broadcast or pad
        w = w.reshape(-1)
        if w.numel() < n_class:
            w = torch.cat([w, torch.ones(n_class - w.numel())])
    if active_classes is not None:
        mask = torch.zeros(n_class, dtype=torch.float32)
        for c in active_classes:
            mask[c] = 1.0
        w = w * mask
    return w


# ---------------------------------------------------------------------------
# Tversky
# ---------------------------------------------------------------------------
class TverskyLoss(nn.Module):
    """Multi-class Tversky loss summed over ACTIVE foreground classes only.

    Operates on logits (B,C,D,H,W) and integer targets (B,D,H,W).
    """

    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-3, ignore_index=-1,
                 active_classes: Optional[List[int]] = None):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth
        self.ignore_index = ignore_index
        self.active_classes = active_classes  # classes to sum over (excl. bg)

    def forward(self, logits, target):
        n_class = logits.shape[1]
        probs = _softmax_probs(logits)
        valid = (target != self.ignore_index)
        t = target.clone()
        t[~valid] = 0
        oh = F.one_hot(t, num_classes=n_class).permute(0, 4, 1, 2, 3).float()
        mask = valid.unsqueeze(1).float()
        probs = probs * mask
        oh = oh * mask

        dims = (0, 2, 3, 4)
        tp = (probs * oh).sum(dims)
        fp = (probs * (1 - oh)).sum(dims)
        fn = ((1 - probs) * oh).sum(dims)
        ti = (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)

        # Only sum over active foreground classes; ignore dead heads.
        if self.active_classes is not None:
            idx = torch.as_tensor(self.active_classes, device=ti.device, dtype=torch.long)
            # exclude background (class 0) from the loss sum
            fg = idx[idx != 0]
            n_active = int(fg.numel())
            return n_active - ti[fg].sum()
        # default: sum over all classes except background (index 0)
        fg_idx = list(range(1, n_class))
        return float(len(fg_idx)) - ti[fg_idx].sum()


class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=2.0, smooth=1e-3, ignore_index=-1,
                 active_classes: Optional[List[int]] = None):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth
        self.ignore_index = ignore_index
        self.active_classes = active_classes

    def forward(self, logits, target):
        n_class = logits.shape[1]
        probs = _softmax_probs(logits)
        valid = (target != self.ignore_index)
        t = target.clone()
        t[~valid] = 0
        oh = F.one_hot(t, num_classes=n_class).permute(0, 4, 1, 2, 3).float()
        mask = valid.unsqueeze(1).float()
        probs = probs * mask
        oh = oh * mask

        dims = (0, 2, 3, 4)
        tp = (probs * oh).sum(dims)
        fp = (probs * (1 - oh)).sum(dims)
        fn = ((1 - probs) * oh).sum(dims)
        ti = (tp + self.smooth) / (tp + self.alpha * fn + self.beta * fp + self.smooth)
        focal = torch.pow((1.0 - ti), self.gamma)

        if self.active_classes is not None:
            idx = torch.as_tensor(self.active_classes, device=focal.device, dtype=torch.long)
            fg = idx[idx != 0]
            return focal[fg].mean()
        fg_idx = list(range(1, n_class))
        return focal[fg_idx].mean()


# ---------------------------------------------------------------------------
# Dice
# ---------------------------------------------------------------------------
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-3, ignore_index=-1, active_classes: Optional[List[int]] = None):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index
        self.active_classes = active_classes

    def forward(self, logits, target):
        n_class = logits.shape[1]
        probs = _softmax_probs(logits)
        valid = (target != self.ignore_index)
        t = target.clone()
        t[~valid] = 0
        oh = F.one_hot(t, num_classes=n_class).permute(0, 4, 1, 2, 3).float()
        mask = valid.unsqueeze(1).float()
        probs = probs * mask
        oh = oh * mask
        dims = (0, 2, 3, 4)
        inter = (probs * oh).sum(dims)
        denom = (probs + oh).sum(dims)
        dice = (2 * inter + self.smooth) / (denom + self.smooth)
        if self.active_classes is not None:
            idx = torch.as_tensor(self.active_classes, device=dice.device, dtype=torch.long)
            fg = idx[idx != 0]
            return 1.0 - dice[fg].mean()
        fg_idx = list(range(1, n_class))
        return 1.0 - dice[fg_idx].mean()


# ---------------------------------------------------------------------------
# Combined CE + Tversky
# ---------------------------------------------------------------------------
class CETverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, ce_weight=1.0, tversky_weight=1.0,
                 smooth=1e-3, ignore_index=-1, class_weights=None,
                 active_classes: Optional[List[int]] = None,
                 n_class: int = 8):
        super().__init__()
        self.ce_weight = ce_weight
        self.tversky_weight = tversky_weight
        self.tversky = TverskyLoss(alpha, beta, smooth, ignore_index, active_classes)
        w = _resolve_class_weights(class_weights, n_class, active_classes)
        self.ce = nn.CrossEntropyLoss(
            weight=w,
            ignore_index=ignore_index,
        )
        self._weight_tensor = w

    def forward(self, logits, target):
        l_ce = self.ce(logits, target)
        l_tv = self.tversky(logits, target)
        return self.ce_weight * l_ce + self.tversky_weight * l_tv


# ---------------------------------------------------------------------------
# Factory
# ---------------------------------------------------------------------------
def build_loss(cfg, n_class: int = 8) -> nn.Module:
    """Build a loss from a LossCfg dataclass.

    ``n_class`` must match the model's output channel count so the CE class
    weight vector is sized correctly.
    """
    cw = cfg.class_weights
    weights = None
    if isinstance(cw, list):
        weights = [float(x) for x in cw]
    active = getattr(cfg, "active_classes", None)

    if cfg.type == "ce_tversky":
        return CETverskyLoss(
            alpha=cfg.alpha, beta=cfg.beta, ce_weight=cfg.ce_weight,
            tversky_weight=cfg.tversky_weight, smooth=cfg.smooth,
            ignore_index=cfg.ignore_index, class_weights=weights,
            active_classes=active, n_class=n_class,
        )
    if cfg.type == "tversky":
        return TverskyLoss(cfg.alpha, cfg.beta, cfg.smooth, cfg.ignore_index, active)
    if cfg.type == "focal_tversky":
        return FocalTverskyLoss(cfg.alpha, cfg.beta, cfg.gamma, cfg.smooth, cfg.ignore_index, active)
    if cfg.type == "dice":
        return DiceLoss(cfg.smooth, cfg.ignore_index, active)
    if cfg.type == "ce":
        w = _resolve_class_weights(weights, n_class, active)
        return nn.CrossEntropyLoss(weight=w, ignore_index=cfg.ignore_index)
    raise ValueError(f"Unknown loss type '{cfg.type}'")



### Metrics (per-class P/R/F1, mIoU)

In [ ]:
"""Segmentation metrics (per-class precision/recall/F1, macro-F1, mIoU).

Macro averages are computed over *active* classes only (background + scored
particles), excluding dead heads (e.g. membrane when not trained), so the
checkpoint-selection metric reflects real task performance rather than being
dragged down by phantom channels.
"""
from __future__ import annotations



from typing import List, Optional

import numpy as np
import torch


@torch.no_grad()
def segmentation_metrics(
    logits: torch.Tensor,
    target: torch.Tensor,
    n_class: int,
    ignore_index: int = -1,
    active_classes: Optional[List[int]] = None,
) -> dict:
    """Compute per-class precision/recall/F1 and macro averages from a batch.

    logits: (B,C,D,H,W)  target: (B,D,H,W) integer class indices
    active_classes: classes to include in the *macro* average. If None, all
        classes 0..n_class-1 are used. Per-class arrays always have length
        n_class; only the macro is restricted.
    """
    pred = logits.argmax(1)
    valid = target != ignore_index
    pred = pred[valid]
    tgt = target[valid]

    tp = torch.zeros(n_class, dtype=torch.long, device=logits.device)
    fp = torch.zeros(n_class, dtype=torch.long, device=logits.device)
    fn = torch.zeros(n_class, dtype=torch.long, device=logits.device)

    for c in range(n_class):
        tp[c] = ((pred == c) & (tgt == c)).sum()
        fp[c] = ((pred == c) & (tgt != c)).sum()
        fn[c] = ((pred != c) & (tgt == c)).sum()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)

    classes = active_classes if active_classes is not None else list(range(n_class))
    cls_idx = torch.as_tensor(classes, device=f1.device, dtype=torch.long)
    macro_f1 = float(f1[cls_idx].mean().item())
    macro_iou = float(iou[cls_idx].mean().item())

    return {
        "precision": precision.cpu().numpy(),
        "recall": recall.cpu().numpy(),
        "f1": f1.cpu().numpy(),
        "iou": iou.cpu().numpy(),
        "macro_f1": macro_f1,
        "macro_iou": macro_iou,
    }


class MetricAccumulator:
    """Running mean of segmentation metrics across batches."""

    def __init__(self, n_class: int, active_classes: Optional[List[int]] = None):
        self.n_class = n_class
        self.active_classes = active_classes
        self.reset()

    def reset(self):
        self._f1 = []
        self._iou = []
        self._prec = []
        self._rec = []
        self._loss = []

    def update(self, metrics: dict, loss: float):
        self._f1.append(metrics["f1"])
        self._iou.append(metrics["iou"])
        self._prec.append(metrics["precision"])
        self._rec.append(metrics["recall"])
        self._loss.append(loss)

    def compute(self) -> dict:
        if not self._f1:
            return {"macro_f1": 0.0, "macro_iou": 0.0, "loss": 0.0,
                    "f1": np.zeros(self.n_class), "iou": np.zeros(self.n_class),
                    "precision": np.zeros(self.n_class), "recall": np.zeros(self.n_class)}
        f1 = np.stack(self._f1).mean(0)
        iou = np.stack(self._iou).mean(0)
        prec = np.stack(self._prec).mean(0)
        rec = np.stack(self._rec).mean(0)
        classes = self.active_classes if self.active_classes is not None else list(range(self.n_class))
        cls_idx = np.asarray(classes, dtype=np.int64)
        return {
            "f1": f1,
            "iou": iou,
            "precision": prec,
            "recall": rec,
            "macro_f1": float(f1[cls_idx].mean()),
            "macro_iou": float(iou[cls_idx].mean()),
            "loss": float(np.mean(self._loss)),
        }



### Inference (sliding-window segmentation)

In [ ]:
"""Sliding-window 3D inference producing per-class scoremaps + labelmap."""
from __future__ import annotations



import time
from typing import Tuple

import numpy as np
import torch



def _cosine_window_3d(d: int, device, dtype) -> torch.Tensor:
    """3D cosine (Hann) window for smooth overlap blending. Returns (d,d,d)."""
    k1 = torch.sin(torch.linspace(0, np.pi, d, device=device, dtype=dtype)) ** 2
    # separable 3D window = outer product along each axis
    w = torch.einsum("i,j,k->ijk", k1, k1, k1)
    return (w / w.max())


@torch.no_grad()
def segment_volume(
    model: torch.nn.Module,
    volume: np.ndarray,  # (Z,Y,X)
    patch_size: int,
    overlap: int,
    pcrop: int,
    n_class: int,
    batch_patches: int = 4,
    amp: bool = True,
    device: torch.device | None = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Segment a full tomogram with sliding-window inference.

    Returns:
        labelmap: (Z,Y,X) uint8 argmax classes
        scoremap: (C,Z,Y,X) float32 softmax probabilities
    """
    log = get_logger("inference")
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()

    P = patch_size
    p = P // 2
    lcrop = p - pcrop
    step = P - overlap

    # Normalize and pad
    vol = (volume.astype(np.float32) - volume.mean()) / (volume.std() + 1e-8)
    vol = np.pad(vol, pcrop, mode="constant", constant_values=0)
    dim = vol.shape  # (Z,Y,X) padded

    # patch centers along each axis
    def centers(d):
        c = list(range(p, d - p, step))
        if not c:
            c = [p]
        if c[-1] < d - p:
            c.append(d - p)
        return c

    cz = centers(dim[0])
    cy = centers(dim[1])
    cx = centers(dim[2])
    Npatch = len(cz) * len(cy) * len(cx)
    log.info(f"Segmenting {Npatch} patches (P={P}, overlap={overlap}, pcrop={pcrop})...")

    pred = np.zeros((n_class,) + dim, dtype=np.float32)
    norm = np.zeros(dim, dtype=np.float32)
    window = _cosine_window_3d(P - 2 * pcrop, device, torch.float32).cpu().numpy()

    patches = []
    coords = []
    count = 0
    t0 = time.time()

    for z in cz:
        for y in cy:
            for x in cx:
                patch = vol[z - p:z + p, y - p:y + p, x - p:x + p]
                patches.append(patch)
                coords.append((z, y, x))
                if len(patches) == batch_patches:
                    _flush(model, patches, coords, pred, norm, window, P, p, lcrop,
                           pcrop, n_class, amp, device)
                    patches.clear()
                    coords.clear()
                count += 1
                if count % 20 == 0:
                    log.info(f"  patch {count}/{Npatch} ({(time.time()-t0):.0f}s)")

    if patches:
        _flush(model, patches, coords, pred, norm, window, P, p, lcrop,
               pcrop, n_class, amp, device)

    # normalize overlaps
    norm[norm == 0] = 1.0
    pred = pred / norm[None]
    # unpad
    pred = pred[:, pcrop:-pcrop, pcrop:-pcrop, pcrop:-pcrop]
    labelmap = np.argmax(pred, axis=0).astype(np.uint8)
    log.info(f"Segmentation done in {time.time()-t0:.0f}s")
    return labelmap, pred


def _flush(model, patches, coords, pred, norm, window, P, p, lcrop, pcrop,
           n_class, amp, device):
    batch = np.stack(patches)[:, None]  # (B,1,P,P,P)
    t = torch.from_numpy(batch).float().to(device)
    with torch.amp.autocast("cuda", enabled=amp and device.type == "cuda"):
        out = model(t)
    probs = torch.softmax(out.float(), dim=1).cpu().numpy()
    win = window[None]  # (1,winD,winD,winD) for broadcasting over classes
    d = P - 2 * pcrop  # valid region side length
    for i, (z, y, x) in enumerate(coords):
        # central valid region of the prediction (crop pcrop from each side)
        p = probs[i, :, pcrop:pcrop + d, pcrop:pcrop + d, pcrop:pcrop + d]
        # valid region starts at lcrop within the padded volume
        z0, y0, x0 = z - lcrop, y - lcrop, x - lcrop
        pred[:, z0:z0 + d, y0:y0 + d, x0:x0 + d] += p * win
        # win[0] is the (d,d,d) 3D window (indexes the leading singleton dim),
        # so this accumulates the full 3D cosine window into norm, matching pred.
        norm[z0:z0 + d, y0:y0 + d, x0:x0 + d] += win[0]


def run_inference(cfg: Config, weights_path: str, tomo_ids=None) -> None:
    """Segment all (or given) tomograms and write labelmaps (+ optional scoremaps)."""
    log = get_logger("inference")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = build_model(
        cfg.model.name, in_channels=cfg.model.in_channels,
        n_class=cfg.model.n_class, filters=cfg.model.filters,
        dropout=cfg.model.dropout,
    )
    load_checkpoint(weights_path, model, map_location=device)
    model = model.to(device).eval()

    if tomo_ids is None:
        tomo_ids = copick_io.list_runs(cfg.data.copick_config)
    log.info(f"Segmenting {len(tomo_ids)} tomograms: {tomo_ids}")

    for tid in tomo_ids:
        log.info(f"=== {tid} ===")
        tomo = copick_io.get_tomogram(
            cfg.data.copick_config, tid, cfg.data.voxel_size, cfg.data.tomo_algorithm
        )[:]
        labelmap, scoremap = segment_volume(
            model, tomo,
            patch_size=cfg.inference.patch_size,
            overlap=cfg.inference.overlap,
            pcrop=cfg.inference.pcrop,
            n_class=cfg.model.n_class,
            batch_patches=cfg.inference.batch_patches,
            amp=cfg.inference.amp,
            device=device,
        )
        copick_io.write_ome_zarr_segmentation(
            cfg.data.copick_config, tid, labelmap, cfg.data.voxel_size,
            name=cfg.inference.segmentation_name,
            user_id=cfg.inference.user_id,
            session_id=cfg.inference.session_id,
        )
        if cfg.inference.write_scoremap:
            copick_io.write_ome_zarr_scoremap(
                cfg.data.copick_config, tid, scoremap, cfg.data.voxel_size,
                name=cfg.inference.scoremap_name,
                user_id=cfg.inference.user_id,
                session_id=cfg.inference.session_id,
                tomo_type=cfg.data.tomo_algorithm,
            )



### Localization (segmentation → coordinates, optional)

In [ ]:
"""Optional: convert segmentations to particle coordinates (connected components).

The stored segmentation carries *contiguous model-class indices* (see
``labels.py``); this module converts them back to copick object names via
``CLASS_TO_LABEL`` / ``CLASS_TO_NAME`` for writing picks.
"""
from __future__ import annotations



import json
import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
from scipy import ndimage



def _remove_duplicates(coords: np.ndarray, threshold: float) -> np.ndarray:
    if coords.shape[0] <= 1:
        return coords
    from scipy.spatial import cKDTree

    tree = cKDTree(coords[:, :3])
    pairs = tree.query_pairs(threshold)
    drop = set()
    for a, b in pairs:
        # keep the first; mark the second for removal (lower score later)
        drop.add(b)
    keep = sorted(set(range(coords.shape[0])) - drop)
    return coords[keep]


def _euler_to_matrix(rot: float, tilt: float, psi: float) -> list:
    from scipy.spatial.transform import Rotation as R

    r = R.from_euler("zyz", [rot, tilt, psi], degrees=True).as_matrix()
    m = np.zeros((4, 4))
    m[:3, :3] = r
    m[3, 3] = 1.0
    return np.round(m, 3).tolist()


def _write_copick_picks(name, tomo_id, coords, path_output, user_id, session_id):
    """Write a copick JSON pick file (coords in Angstroms, x/y/z)."""
    pts = []
    for row in coords:
        x, y, z = float(row[0]), float(row[1]), float(row[2])
        score = float(row[3]) if row.shape[0] > 3 else 1.0
        pts.append({
            "location": {"x": x, "y": y, "z": z},
            "transformation_": _euler_to_matrix(0, 0, 0),
            "instance_id": 0,
            "score": score,
        })
    data = {
        "pickable_object_name": name,
        "user_id": user_id,
        "session_id": session_id,
        "run_name": tomo_id,
        "voxel_spacing": None,
        "unit": "angstrom",
        "trust_orientation": "false",
        "points": pts,
    }
    out_dir = Path(path_output) / tomo_id / "Picks"
    out_dir.mkdir(parents=True, exist_ok=True)
    fname = out_dir / f"{user_id}_{session_id}_{name}.json"
    with open(fname, "w") as f:
        json.dump(data, f, indent=2)


def _object_radius_voxels(config_path: str, object_name: str, voxel_size: float) -> float:
    objs = {o["name"]: o for o in copick_io.get_pickable_objects(config_path)}
    o = objs.get(object_name)
    if not o or not o.get("radius"):
        return 0.0
    return float(o["radius"]) / float(voxel_size)


def localize_segmentations(
    cfg: Config,
    tomo_ids: Optional[List[str]] = None,
    segmentation_name: Optional[str] = None,
) -> None:
    """Convert stored segmentations into particle coordinate picks.

    Iterates over *particle* model-class indices only (apo-ferritin=1 through
    virus-like-particle=6), so apo-ferritin is no longer dropped. Background
    (0) and membrane (7) are skipped automatically.
    """
    log = get_logger("localize")
    root = copick_io.get_copick_root(cfg.data.copick_config)
    if tomo_ids is None:
        tomo_ids = [r.name for r in root.runs]

    seg_name = segmentation_name or cfg.inference.segmentation_name
    out_overlay = cfg.inference.out_overlay
    for tid in tomo_ids:
        log.info(f"Localizing {tid}")
        try:
            labelmap = copick_io.get_segmentation(
                cfg.data.copick_config, tid, name=seg_name,
                user_id=cfg.inference.user_id, session_id=cfg.inference.session_id,
            )[:]
        except Exception as e:
            log.warning(f"  no segmentation '{seg_name}' for {tid}: {e}")
            continue

        # Iterate over the six particle classes (1..6).
        for cls in PARTICLE_CLASSES:
            name = CLASS_TO_NAME.get(cls)
            if name is None:
                continue
            r_vox = _object_radius_voxels(cfg.data.copick_config, name, cfg.data.voxel_size)
            if r_vox <= 0:
                continue

            lbl_objs, _ = ndimage.label(labelmap == cls)
            sizes = np.bincount(lbl_objs.ravel())
            min_size = (4 / 3) * np.pi * (r_vox ** 3) * cfg.localize.min_protein_size
            valid = np.where(sizes > min_size)[0]
            valid = valid[valid != 0]  # drop background label 0

            coords = []
            for oid in valid:
                com = ndimage.center_of_mass(lbl_objs == oid)  # (z,y,x)
                x, y, z = com[2], com[1], com[0]
                coords.append((x, y, z, 1.0))

            if not coords:
                continue
            coords = np.array(coords, dtype=np.float32)
            threshold = np.ceil(r_vox * 3)
            coords = _remove_duplicates(coords, threshold)
            # voxel -> angstrom
            coords[:, :3] *= cfg.data.voxel_size

            if cfg.localize.write_copick_picks:
                _write_copick_picks(
                    name, tid, coords, out_overlay,
                    cfg.localize.picks_user_id, cfg.localize.picks_session_id,
                )
            log.info(f"  {name}: {coords.shape[0]} picks")



### Training loop (AMP, DataParallel, cosine LR, checkpointing, auto-resume, per-epoch full-slice inline viz)

In [ ]:
"""Training loop for 3D CryoET segmentation (PyTorch, AMP, multi-GPU).

Validation / model selection is driven by a *full-volume* sliding-window pass
over a held-out tomogram (via ``segment_volume``), not patch-level F1 on random
crops, so the chosen checkpoint reflects end-task segmentation quality. Only
*active* classes (background + scored particles) participate in the macro-F1
used for selection; dead heads (e.g. membrane when not trained) are excluded.
"""
from __future__ import annotations



import os
import math
import time
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
from torch.utils.data import DataLoader

try:
    from torch.utils.tensorboard import SummaryWriter
except Exception:  # tensorboard optional; its import chain (TF/pyOpenSSL) can be broken on Kaggle
    SummaryWriter = None



def _active_classes_for(cfg: Config) -> List[int]:
    """Classes that carry target signal (background + particles)."""
    if cfg.loss.active_classes is not None:
        return list(cfg.loss.active_classes)
    return list(ACTIVE_CLASSES)


def _inverse_freq_weights(picks, n_class: int, active_classes: List[int]) -> list:
    """Inverse-frequency class weights over ACTIVE classes only.

    Inactive classes get zero weight so dead heads (e.g. membrane) don't pull
    the model. Background (class 0) is down-weighted.
    """
    counts = np.zeros(n_class, dtype=np.float64)
    for *_p, cls in picks:
        counts[cls] += 1
    counts[counts == 0] = 1.0
    w = 1.0 / counts
    # normalize over active classes so the mean active weight is 1.0
    active = np.asarray(active_classes, dtype=np.int64)
    w_active_mean = w[active].mean() if active.size else 1.0
    w = w / max(w_active_mean, 1e-8)
    # zero out inactive classes
    mask = np.zeros(n_class, dtype=np.float64)
    mask[active] = 1.0
    w = w * mask
    # down-weight background relative to the mean particle weight
    if 0 in active:
        particle_w = w[[c for c in active if c != 0]]
        w[0] = 0.1 * (particle_w.mean() if particle_w.size else 1.0)
    return w.tolist()


def _build_optimizer(cfg: Config, model: torch.nn.Module) -> torch.optim.Optimizer:
    params = [p for p in model.parameters() if p.requires_grad]
    if cfg.train.optimizer.lower() == "adamw":
        return torch.optim.AdamW(
            params, lr=cfg.train.lr, betas=tuple(cfg.train.betas),
            eps=cfg.train.eps, weight_decay=cfg.train.weight_decay,
        )
    if cfg.train.optimizer.lower() == "adam":
        return torch.optim.Adam(
            params, lr=cfg.train.lr, betas=tuple(cfg.train.betas),
            eps=cfg.train.eps, weight_decay=cfg.train.weight_decay,
        )
    if cfg.train.optimizer.lower() == "sgd":
        return torch.optim.SGD(
            params, lr=cfg.train.lr, momentum=0.9, weight_decay=cfg.train.weight_decay
        )
    raise ValueError(f"Unknown optimizer '{cfg.train.optimizer}'")


def _build_scheduler(cfg: Config, optimizer, steps_per_epoch: int):
    if cfg.train.scheduler == "cosine":
        from torch.optim.lr_scheduler import CosineAnnealingLR

        return CosineAnnealingLR(optimizer, T_max=cfg.train.epochs,
                                 eta_min=cfg.train.min_lr)
    if cfg.train.scheduler == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.75, patience=6, min_lr=cfg.train.min_lr
        )
    return None


def _warmup_lr(optimizer, step, warmup_steps, base_lr):
    if warmup_steps <= 0:
        return
    lr = base_lr * min(1.0, step / max(1, warmup_steps))
    for pg in optimizer.param_groups:
        pg["lr"] = lr


def _unwrap(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


@torch.no_grad()
def full_volume_metrics(
    model, cfg, tomo_id, active_classes, device, log
) -> dict:
    """Run sliding-window inference on one full tomogram and score it against
    the stored target. Returns per-class + macro F1/IoU over active classes.

    This is a more honest checkpoint-selection signal than patch-level F1 on
    random crops, because it measures end-to-end segmentation quality on a
    realistic volume.
    """
    tomo = copick_io.get_tomogram(
        cfg.data.copick_config, tomo_id, cfg.data.voxel_size, cfg.data.tomo_algorithm
    )[:]
    try:
        tgt = copick_io.get_segmentation(
            cfg.data.copick_config, tomo_id,
            name=cfg.data.target_name,
            user_id=cfg.data.target_user_id,
            session_id=cfg.data.target_session_id,
        )[:]
    except Exception as e:
        log.warning(f"  no target for {tomo_id}: {e}; skipping full-volume eval")
        return {"macro_f1": 0.0, "macro_iou": 0.0}

    labelmap, _ = segment_volume(
        _unwrap(model), tomo,
        patch_size=cfg.inference.patch_size,
        overlap=cfg.inference.overlap,
        pcrop=cfg.inference.pcrop,
        n_class=cfg.model.n_class,
        batch_patches=cfg.inference.batch_patches,
        amp=cfg.inference.amp,
        device=device,
    )
    pred = torch.from_numpy(labelmap.astype(np.int64))[None].to(device)        # (1,Z,Y,X)
    target = torch.from_numpy(tgt.astype(np.int64))[None].to(device)          # (1,Z,Y,X)
    # segmentation_metrics expects logits (B,C,D,H,W) and target (B,D,H,W).
    onehot = torch.nn.functional.one_hot(pred, cfg.model.n_class).permute(0, 4, 1, 2, 3)  # (1,C,Z,Y,X)
    onehot = onehot.float() * 20.0  # large logits so argmax == pred
    m = segmentation_metrics(onehot, target, cfg.model.n_class, cfg.loss.ignore_index, active_classes)
    return m


def train(cfg: Config) -> str:
    """Run training. Returns path to the best checkpoint."""
    log = get_logger("train")
    seed_everything(cfg.train.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu = torch.cuda.device_count()
    log.info(f"Device: {device} | GPUs: {n_gpu}")
    if n_gpu > 1:
        log.info(f"Using DataParallel across {n_gpu} GPUs")

    out_dir = Path(cfg.train.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg.save_yaml(out_dir / "config_used.yaml")

    # ---- Data: split tomo IDs -----------------------------------------
    all_runs = copick_io.list_runs(cfg.data.copick_config)
    if cfg.data.train_tomo_ids and cfg.data.valid_tomo_ids:
        train_ids = cfg.data.train_tomo_ids
        valid_ids = cfg.data.valid_tomo_ids
    else:
        train_ids, valid_ids, _test = split_utils.split_runs(
            all_runs, cfg.data.train_ratio, cfg.data.val_ratio, cfg.data.test_ratio,
            seed=cfg.train.seed,
        )
        if len(_test) > len(valid_ids):
            valid_ids, _test = _test, valid_ids
    log.info(f"Train tomos ({len(train_ids)}): {train_ids}")
    log.info(f"Valid tomos ({len(valid_ids)}): {valid_ids}")

    # ---- Active classes + class weights ------------------------------
    active_classes = _active_classes_for(cfg)
    log.info(f"Active classes (loss/metric): {active_classes}")
    cfg.loss.active_classes = active_classes

    train_picks = index_picks(cfg.data.copick_config, train_ids, None, cfg.data.voxel_size)
    if cfg.loss.class_weights == "inverse":
        weights = _inverse_freq_weights(train_picks, cfg.model.n_class, active_classes)
        log.info(f"Class weights (inverse-freq, active-only): {[round(w,4) for w in weights]}")
        cfg.loss.class_weights = weights

    # ---- Model / loss / optim ----------------------------------------
    model = build_model(
        cfg.model.name, in_channels=cfg.model.in_channels,
        n_class=cfg.model.n_class, filters=cfg.model.filters,
        dropout=cfg.model.dropout,
    )
    if n_gpu > 1:
        model = torch.nn.DataParallel(model)
    model = model.to(device)
    criterion = build_loss(cfg.loss, n_class=cfg.model.n_class).to(device)
    optimizer = _build_optimizer(cfg, model)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.train.amp and device.type == "cuda")
    scheduler = _build_scheduler(cfg, optimizer, cfg.train.steps_per_epoch)

    start_epoch = 0
    best_f1 = -1.0

    # ---- Resume logic ----
    resume_path = cfg.train.resume
    if resume_path is None:
        best_path = out_dir / "net_weights_BEST.pt"
        last_path = out_dir / "net_weights_LAST.pt"
        if best_path.exists():
            resume_path = str(best_path)
            log.info(f"Found existing BEST weights -> resuming from {resume_path}")
        elif last_path.exists():
            resume_path = str(last_path)
            log.info(f"Found existing LAST weights -> resuming from {resume_path}")

    if resume_path is not None and Path(resume_path).exists():
        ckpt = load_checkpoint(resume_path, model, optimizer, scheduler, scaler, map_location=device)
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_f1 = float(ckpt.get("best_metric") or -1.0)
        log.info(f"Resumed from {resume_path} -> start_epoch={start_epoch} best_f1={best_f1:.4f}")
    elif resume_path is not None:
        log.warning(f"Resume path {resume_path} does not exist; training from scratch.")
    else:
        log.info("No existing checkpoint found; training from scratch.")

    # ---- Datasets / loaders ------------------------------------------
    augment = Augment3D()
    train_ds = CryoETPatchDataset(
        cfg.data.copick_config, train_ids, cfg.data.dim_in, cfg.train.batch_size,
        steps=cfg.train.steps_per_epoch, l_rnd=cfg.data.l_rnd,
        background_ratio=cfg.data.background_ratio,
        voxel_size=cfg.data.voxel_size, tomo_algorithm=cfg.data.tomo_algorithm,
        target_name=cfg.data.target_name, target_user_id=cfg.data.target_user_id,
        target_session_id=cfg.data.target_session_id, augment=augment,
        n_sub_epoch=cfg.data.n_sub_epoch, sample_size=cfg.data.sample_size,
        seed=cfg.train.seed,
    )
    # Patch-level validation loader is kept for a quick per-epoch sanity loss,
    # but checkpoint selection uses the full-volume pass below.
    valid_ds = CryoETPatchDataset(
        cfg.data.copick_config, valid_ids, cfg.data.dim_in, cfg.train.batch_size,
        steps=cfg.train.steps_per_valid, l_rnd=cfg.data.l_rnd,
        background_ratio=cfg.data.background_ratio,
        voxel_size=cfg.data.voxel_size, tomo_algorithm=cfg.data.tomo_algorithm,
        target_name=cfg.data.target_name, target_user_id=cfg.data.target_user_id,
        target_session_id=cfg.data.target_session_id, augment=None,
        n_sub_epoch=cfg.data.n_sub_epoch, sample_size=cfg.data.sample_size,
        seed=cfg.train.seed + 1,
    )

    def _loader(ds, shuffle=False):
        return DataLoader(
            ds, batch_size=None, num_workers=cfg.train.n_workers,
            pin_memory=cfg.train.pin_memory and device.type == "cuda",
            worker_init_fn=lambda wid: None,
        )

    # Held-out tomogram for full-volume model selection.
    vol_eval_id = valid_ids[0] if valid_ids else (train_ids[0] if train_ids else None)

    # ---- TensorBoard / history ----------------------------------------
    writer = SummaryWriter(log_dir=str(out_dir / "tensorboard_logs")) if SummaryWriter else None
    history = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "val_macro_iou": [],
               "vol_macro_f1": [], "lr": []}

    warmup_steps = cfg.train.warmup_epochs * cfg.train.steps_per_epoch
    global_step = start_epoch * cfg.train.steps_per_epoch

    for epoch in range(start_epoch, cfg.train.epochs):
        model.train()
        t0 = time.time()
        train_loader = _loader(train_ds)
        running_loss = 0.0
        for step, (img, lbl) in enumerate(train_loader):
            img = img.to(device, non_blocking=True)
            lbl = lbl.to(device, non_blocking=True)

            if global_step < warmup_steps:
                _warmup_lr(optimizer, global_step, warmup_steps, cfg.train.lr)

            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=cfg.train.amp and device.type == "cuda"):
                out = model(img)
                loss = criterion(out, lbl)
            scaler.scale(loss).backward()
            if cfg.train.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.train.grad_clip)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.item())
            global_step += 1

            if step % 20 == 0:
                log.info(f"E{epoch} S{step}/{cfg.train.steps_per_epoch} loss={float(loss.item()):.4f} lr={optimizer.param_groups[0]['lr']:.2e}")

        train_loss = running_loss / max(1, cfg.train.steps_per_epoch)

        # ---- Quick patch-level validation loss (sanity) ----
        model.eval()
        acc = MetricAccumulator(cfg.model.n_class, active_classes)
        with torch.no_grad():
            for img, lbl in _loader(valid_ds):
                img = img.to(device, non_blocking=True)
                lbl = lbl.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=cfg.train.amp and device.type == "cuda"):
                    out = model(img)
                    loss = criterion(out, lbl)
                m = segmentation_metrics(out.float(), lbl, cfg.model.n_class, cfg.loss.ignore_index, active_classes)
                acc.update(m, float(loss.item()))
        vm = acc.compute()

        # ---- Full-volume validation (drives checkpoint selection) ----
        # Full-volume inference slides over an entire tomogram and is much
        # costlier than patch-level validation. Run it every `vol_eval_every`
        # epochs (and on the final epoch); on other epochs fall back to patch
        # F1 so checkpoint selection still works.
        is_last_epoch = (epoch == cfg.train.epochs - 1)
        do_vol_eval = (vol_eval_id is not None) and (
            (epoch + 1) % max(1, cfg.train.vol_eval_every) == 0 or is_last_epoch
        )
        vol_f1 = vm["macro_f1"]
        vol_iou = vm["macro_iou"]
        if do_vol_eval:
            try:
                vm_vol = full_volume_metrics(model, cfg, vol_eval_id, active_classes, device, log)
                vol_f1 = vm_vol["macro_f1"]
                vol_iou = vm_vol["macro_iou"]
                log.info(
                    f"[Epoch {epoch}] train_loss={train_loss:.4f} patch_loss={vm['loss']:.4f} "
                    f"patchF1={vm['macro_f1']:.4f} | VOLUME macroF1={vol_f1:.4f} "
                    f"mIoU={vol_iou:.4f} ({time.time()-t0:.0f}s)"
                )
            except Exception as e:
                log.warning(f"  full-volume eval failed: {e}; falling back to patch F1")
                log.info(
                    f"[Epoch {epoch}] train_loss={train_loss:.4f} val_loss={vm['loss']:.4f} "
                    f"macroF1={vm['macro_f1']:.4f} mIoU={vm['macro_iou']:.4f} ({time.time()-t0:.0f}s)"
                )
        else:
            log.info(
                f"[Epoch {epoch}] train_loss={train_loss:.4f} val_loss={vm['loss']:.4f} "
                f"macroF1={vm['macro_f1']:.4f} mIoU={vm['macro_iou']:.4f} "
                f"(patch-only; full-vol eval every {cfg.train.vol_eval_every} eps) "
                f"({time.time()-t0:.0f}s)"
            )

        # ---- LR schedule ----
        if scheduler is not None:
            if cfg.train.scheduler == "plateau":
                scheduler.step(vol_f1)
            else:
                scheduler.step()

        # ---- Logging / checkpointing ----
        history["train_loss"].append(train_loss)
        history["val_loss"].append(vm["loss"])
        history["val_macro_f1"].append(vm["macro_f1"])
        history["val_macro_iou"].append(vm["macro_iou"])
        history["vol_macro_f1"].append(vol_f1)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        import json as _json
        _json.dump(history, open(out_dir / "history.json", "w"))

        if writer is not None:
            writer.add_scalar("train/loss", train_loss, epoch)
            writer.add_scalar("valid/patch_loss", vm["loss"], epoch)
            writer.add_scalar("valid/patch_macro_f1", vm["macro_f1"], epoch)
            writer.add_scalar("valid/volume_macro_f1", vol_f1, epoch)
            writer.add_scalar("valid/volume_macro_iou", vol_iou, epoch)
            writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)
            for c in active_classes:
                writer.add_scalar(f"valid/f1_class{c}", vm["f1"][c], epoch)

        ckpt_path = out_dir / "net_weights_LAST.pt"
        save_checkpoint(str(ckpt_path), _unwrap(model),
                        optimizer, scheduler, scaler, epoch, best_f1)

        # Select BEST on the full-volume macro-F1 (the end-task signal).
        if vol_f1 > best_f1:
            best_f1 = vol_f1
            best_path = out_dir / "net_weights_BEST.pt"
            save_checkpoint(str(best_path), _unwrap(model),
                            optimizer, scheduler, scaler, epoch, best_f1)
            log.info(f"  -> new best volume macroF1={best_f1:.4f} saved to {best_path}")

        if (epoch + 1) % max(1, cfg.train.save_every) == 0:
            ep_path = out_dir / f"net_weights_epoch{epoch+1}.pt"
            save_checkpoint(str(ep_path), _unwrap(model),
                            optimizer, scheduler, scaler, epoch, best_f1)

    if writer is not None:
        writer.close()
    log.info(f"Training complete. best volume macroF1={best_f1:.4f}")
    return str(out_dir / "net_weights_BEST.pt")



## 1b. Visualization helpers

A small `vis` helper module (inlined) for rich annotated overlays, montages, and
orthogonal views. Uses the copick class colors so each particle type is consistent
across every figure. These helpers are also used by the per-epoch visualization inside
the training loop. Run this cell once to define the helpers used by later cells.

In [ ]:
"""Visualization helpers for 3D CryoET volumes + segmentations."""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ---- Class metadata (from the copick config) ----
CLASS_NAMES = {
    0: "background", 1: "apo-ferritin", 2: "beta-amylase",
    3: "beta-galactosidase", 4: "ribosome", 5: "thyroglobulin",
    6: "virus-like-particle", 8: "membrane", 9: "background2",
}
# RGBA colors per class (0..255) from the copick config blob
CLASS_RGBA = {
    0: (0,   0,   0,   0),       # background (transparent)
    1: (0,   117, 220, 200),     # apo-ferritin (blue)
    2: (153, 63,  0,   200),      # beta-amylase (brown)
    3: (76,  0,   92,  200),      # beta-galactosidase (purple)
    4: (0,   92,  49,  200),      # ribosome (green)
    5: (43,  206, 72,  200),      # thyroglobulin (lime)
    6: (255, 204, 153, 200),      # vlp (peach)
    8: (100, 100, 100, 200),      # membrane (grey)
    9: (10,  150, 200, 200),      # background2
}

def _label_cmap(labels):
    """Build a ListedColormap + norm covering exactly the labels present."""
    labels = sorted([int(l) for l in labels])
    colors = np.array([CLASS_RGBA.get(l, (255,255,255,200)) for l in labels], dtype=float) / 255.0
    cmap = ListedColormap(colors)
    bounds = np.array(labels + [labels[-1]+1], dtype=float) - 0.5
    from matplotlib.colors import BoundaryNorm
    norm = BoundaryNorm(bounds, cmap.N)
    return cmap, norm, labels

def overlay_slice(tomo_slice, label_slice, alpha=0.55, ax=None, title=None):
    """Show a tomogram slice with a semi-transparent color-coded label overlay."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(tomo_slice, cmap="gray")
    labels_present = np.unique(label_slice)
    labels_present = labels_present[labels_present > 0]
    if len(labels_present) > 0:
        cmap, norm, labels = _label_cmap(labels_present)
        ax.imshow(label_slice, cmap=cmap, norm=norm, alpha=alpha)
    ax.set_title(title or ""); ax.axis("off")
    return ax

def legend_for_labels(labels, ax=None, loc="upper right"):
    """Add a legend listing the label classes present."""
    labels = sorted([int(l) for l in labels if l > 0])
    handles = [Patch(facecolor=np.array(CLASS_RGBA.get(l,(255,255,255,200)))/255.0,
                     edgecolor="white", label=CLASS_NAMES.get(l, f"label {l}"))
               for l in labels]
    if ax is not None:
        leg = ax.legend(handles=handles, loc=loc, fontsize=8, framealpha=0.85)
        leg.get_frame().set_edgecolor("white")
    return handles

def montage(tomo, labels=None, n_slices=8, axis=0, figsize=None, suptitle=None):
    """Row of n_slices through the volume along the given axis (0=z,1=y,2=x).
    If `labels` (a matching label volume) is given, show overlays with legends."""
    dim = tomo.shape
    slc_idx = np.linspace(dim[axis]*0.15, dim[axis]*0.85, n_slices).astype(int)
    cols = n_slices
    fig, axes = plt.subplots(1, cols, figsize=figsize or (2.6*cols, 2.8))
    if cols == 1: axes = [axes]
    for i, idx in enumerate(slc_idx):
        if axis == 0:
            t, l = tomo[idx], (labels[idx] if labels is not None else None)
        elif axis == 1:
            t, l = tomo[:, idx, :], (labels[:, idx, :] if labels is not None else None)
        else:
            t, l = tomo[:, :, idx], (labels[:, :, idx] if labels is not None else None)
        axes[i].imshow(t, cmap="gray")
        if l is not None:
            lp = np.unique(l); lp = lp[lp > 0]
            if len(lp):
                cmap, norm, _ = _label_cmap(lp)
                axes[i].imshow(l, cmap=cmap, norm=norm, alpha=0.55)
        axes[i].set_title(f"{['z','y','x'][axis]}={idx}", fontsize=9)
        axes[i].axis("off")
    if suptitle: fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    return fig

def orthogonal_views(tomo, labels=None, alpha=0.55, suptitle=None):
    """Three orthogonal mid-slice views (XY, XZ, YZ) with optional overlays."""
    zc, yc, xc = np.array(tomo.shape) // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    views = [
        ("XY  (z=%d)" % zc, tomo[zc], (labels[zc] if labels is not None else None)),
        ("XZ  (y=%d)" % yc, tomo[:, yc, :], (labels[:, yc, :] if labels is not None else None)),
        ("YZ  (x=%d)" % xc, tomo[:, :, xc], (labels[:, :, xc] if labels is not None else None)),
    ]
    for ax, (name, t, l) in zip(axes, views):
        ax.imshow(t, cmap="gray")
        if l is not None:
            lp = np.unique(l); lp = lp[lp > 0]
            if len(lp):
                cmap, norm, _ = _label_cmap(lp)
                ax.imshow(l, cmap=cmap, norm=norm, alpha=alpha)
                legend_for_labels(lp, ax=ax)
        ax.set_title(name); ax.axis("off")
    if suptitle: fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    return fig

def class_breakdown(label_vol, ax=None, title="Voxel counts per class"):
    """Bar chart of voxel counts per class in a label volume."""
    unq, cnt = np.unique(label_vol, return_counts=True)
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 4))
    names = [CLASS_NAMES.get(int(u), f"label {int(u)}") for u in unq]
    colors = [np.array(CLASS_RGBA.get(int(u),(200,200,200,200)))[:3]/255.0 for u in unq]
    ax.bar(names, cnt, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_title(title); ax.set_xlabel("class"); ax.set_ylabel("voxel count")
    plt.xticks(rotation=30, ha="right", fontsize=8)
    return ax

def picks_on_slice(tomo_slice, picks_xyz, voxel_size, slice_axis, slice_idx, tol=3, ax=None, title=None):
    """Scatter ground-truth pick coordinates that fall near a given slice onto it.
    picks_xyz: (N,3) in Angstroms. slice_axis: 0=z,1=y,2=x."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8,8))
    ax.imshow(tomo_slice, cmap="gray")
    if picks_xyz is not None and len(picks_xyz):
        vcoords = picks_xyz / voxel_size  # -> voxels (x,y,z)
        for i in range(len(vcoords)):
            x, y, z = vcoords[i]
            if slice_axis == 0 and abs(z - slice_idx) <= tol:
                ax.scatter(x, y, s=40, facecolors="none", edgecolors="red", linewidths=1.5)
            elif slice_axis == 1 and abs(y - slice_idx) <= tol:
                ax.scatter(x, z, s=40, facecolors="none", edgecolors="red", linewidths=1.5)
            elif slice_axis == 2 and abs(x - slice_idx) <= tol:
                ax.scatter(y, z, s=40, facecolors="none", edgecolors="red", linewidths=1.5)
    ax.set_title(title or ""); ax.axis("off")
    return ax

print("Visualization helpers defined: overlay_slice, legend_for_labels, montage,",
      "orthogonal_views, class_breakdown, picks_on_slice")

## 2. Stage 0 — Set up the copick project

Writes `/kaggle/working/copick.config` and copies the train overlay (ground-truth picks)
into `/kaggle/working/overlay` so copick can find both tomograms and annotations.

In [ ]:
import os, json, shutil
from pathlib import Path

INPUT = "/kaggle/input/competitions/czii-cryo-et-object-identification"
config_path = os.path.join(WORK, "copick.config")
overlay_out = os.path.join(WORK, "overlay")

def find_static_root(input_root):
    root = Path(input_root)
    for c in [root/"train"/"static", root/"static", root]:
        if (c/"ExperimentRuns").exists():
            return str(c)
    for p in root.rglob("ExperimentRuns"):
        return str(p.parent)
    raise FileNotFoundError(f"Could not locate static/ExperimentRuns under {input_root}")

def copy_overlay(source_overlay, dest_overlay):
    src = Path(source_overlay); dst = Path(dest_overlay)
    if not src.exists(): return
    for root, _dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        target_dir = dst / rel
        target_dir.mkdir(parents=True, exist_ok=True)
        for f in files:
            new_name = f if f.startswith("curation_0_") else f"curation_0_{f}"
            shutil.copy2(Path(root)/f, target_dir/new_name)

static_root = find_static_root(INPUT)
print("Static root:", static_root)
train_overlay = Path(INPUT)/"train"/"overlay"
if not train_overlay.exists():
    train_overlay = Path(INPUT)/"overlay"
if train_overlay.exists():
    copy_overlay(str(train_overlay), overlay_out)
    print(f"Copied overlay picks {train_overlay} -> {overlay_out}")

config_blob = json.dumps({
    "name": "czii_cryoet_mlchallenge_2024",
    "description": "2024 CZII CryoET ML Challenge training data.",
    "version": "1.0.0",
    "pickable_objects": [
        {"name": "apo-ferritin",        "is_particle": True, "pdb_id": "4V1W", "label": 1, "color": [0,117,220,128],   "radius": 60,  "map_threshold": 0.0418},
        {"name": "beta-amylase",        "is_particle": True, "pdb_id": "1FA2", "label": 2, "color": [153,63,0,128],    "radius": 65,  "map_threshold": 0.035},
        {"name": "beta-galactosidase",  "is_particle": True, "pdb_id": "6X1Q", "label": 3, "color": [76,0,92,128],     "radius": 90,  "map_threshold": 0.0578},
        {"name": "ribosome",            "is_particle": True, "pdb_id": "6EK0", "label": 4, "color": [0,92,49,128],     "radius": 150, "map_threshold": 0.0374},
        {"name": "thyroglobulin",       "is_particle": True, "pdb_id": "6SCJ", "label": 5, "color": [43,206,72,128],   "radius": 130, "map_threshold": 0.0278},
        {"name": "virus-like-particle", "is_particle": True,                  "label": 6, "color": [255,204,153,128], "radius": 135, "map_threshold": 0.201},
        {"name": "membrane",            "is_particle": False,                 "label": 8, "color": [100,100,100,128]},
        {"name": "background",          "is_particle": False,                 "label": 9, "color": [10,150,200,128]}
    ],
    "overlay_root": overlay_out,
    "overlay_fs_args": {"auto_mkdir": True},
    "static_root": static_root
}, indent=4)
with open(config_path, "w") as f:
    f.write(config_blob)
print("Wrote copick config ->", config_path)

## 3. Inspect the dataset

List available runs and pickable objects, confirm tomograms load.

In [ ]:
import copick
root = copick.from_file(config_path)
runs = [r.name for r in root.runs]
print(f"Runs ({len(runs)}):", runs)
print("\nPickable objects:")
for o in root.pickable_objects:
    r = getattr(o, "radius", None)
    print(f"  label={o.label:2d}  is_particle={o.is_particle}  name={o.name:25s} radius={r}")

In [ ]:
# Confirm a tomogram loads and show its shape + pick counts
tid = runs[0]
tomo = get_tomogram(config_path, tid, voxel_size=10, tomo_algorithm="denoised")
print(f"{tid} denoised shape (Z,Y,X):", tomo.shape, "dtype:", tomo.dtype)
for o in root.pickable_objects:
    if o.is_particle:
        coords = get_picks(config_path, tid, o.name)
        print(f"  {o.name}: {coords.shape[0]} picks")
        break

### 3a. Raw tomogram — orthogonal views + montage

Three orthogonal mid-slices (XY / XZ / YZ) and a z-slice montage so you can see the
full 3D context of one tomogram before any annotations.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
tomo_vol = tomo[:]  # materialize once for all viz cells
fig = orthogonal_views(tomo_vol, suptitle=f"{tid} — raw denoised tomogram (orthogonal mid-slices)")
plt.show()

In [ ]:
# Z-slice montage: 8 slices from 15%..85% depth
fig = montage(tomo_vol, n_slices=8, axis=0, suptitle=f"{tid} — raw tomogram z-slices")
plt.show()

In [ ]:
# Intensity histogram + stats (helps pick normalization / thresholds)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(tomo_vol.ravel(), bins=200, color="steelblue", edgecolor="none")
ax.set_yscale("log")
ax.set_title(f"{tid} tomogram intensity histogram (log y)")
ax.set_xlabel("intensity"); ax.set_ylabel("count")
plt.tight_layout(); plt.show()
print(f"intensity: min={tomo_vol.min():.3f} max={tomo_vol.max():.3f} "
      f"mean={tomo_vol.mean():.3f} std={tomo_vol.std():.3f}")

## 4. Stage 1 — Build sphere segmentation targets

Stamps filled spheres at each pick location into a uint8 label volume (per run),
written to the overlay as `pytargets`. This is the training ground truth.

In [ ]:
%%time
build_targets(
    config_path, tomo_ids=None, voxel_size=10,
    out_name="pytargets", out_user_id="pytorch", out_session_id="0",
)
print("Targets built for all runs.")

### 4a. Target visualization — what the model learns to predict

The target is a uint8 volume where each particle pick is stamped as a filled sphere
colored by its class. Below: annotated overlays on the tomogram, a z-slice montage with
overlays, orthogonal views, and a per-class voxel-count bar chart.

In [ ]:
# Annotated overlay: tomogram + sphere targets at mid-z, with class legend
tgt = get_segmentation(config_path, tid, name="pytargets", user_id="pytorch", session_id="0")[:]
z = tomo_vol.shape[0] // 2
fig, ax = plt.subplots(figsize=(9, 9))
overlay_slice(tomo_vol[z], tgt[z], alpha=0.55, ax=ax,
              title=f"{tid} — target overlay z={z} (sphere-stamped picks)")
legend_for_labels(np.unique(tgt), ax=ax)
plt.tight_layout(); plt.show()
print("Unique target labels:", np.unique(tgt))

In [ ]:
# Z-slice montage WITH target overlays — see how spheres distribute through depth
fig = montage(tomo_vol, labels=tgt, n_slices=8, axis=0,
             suptitle=f"{tid} — tomogram + target overlay (z-slices)")
plt.show()

In [ ]:
# Orthogonal views with target overlays
fig = orthogonal_views(tomo_vol, labels=tgt, alpha=0.55,
                       suptitle=f"{tid} — target overlays on 3 orthogonal planes")
plt.show()

In [ ]:
# Per-class voxel breakdown of the target (how many voxels per particle type)
fig, ax = plt.subplots(figsize=(9, 4))
class_breakdown(tgt, ax=ax, title=f"{tid} — target voxel counts per class")
plt.tight_layout(); plt.show()

### 4b. Ground-truth pick locations on tomogram slices

Red circles show the (x,y,z) pick coordinates from the copick JSON, overlaid on the
matching tomogram slice. Compare against the sphere targets above to see how radius
maps a point to a filled ball.

In [ ]:
# Scatter the raw pick coordinates on a mid-z slice for one particle type
import numpy as np, matplotlib.pyplot as plt
target_obj = "ribosome"   # change to any pickable object name
picks_xyz = get_picks(config_path, tid, target_obj)
print(f"{target_obj}: {picks_xyz.shape[0]} picks in {tid}")
z = tomo_vol.shape[0] // 2
fig, ax = plt.subplots(figsize=(9, 9))
picks_on_slice(tomo_vol[z], picks_xyz, voxel_size=10, slice_axis=0, slice_idx=z,
               tol=5, ax=ax, title=f"{tid} z={z} — {target_obj} picks ({picks_xyz.shape[0]} total)")
plt.tight_layout(); plt.show()

## 5. Stage 2 — Train the 3D residual U-Net

Loads the config, builds model/loss/optimizer, runs the training loop with AMP,
class-balanced bootstrap sampling, tomogram pool swaps, cosine LR + warmup,
and best-F1 checkpointing. Logs to TensorBoard under `train_results/tensorboard_logs`.

**Auto-resume:** if `train_results/net_weights_BEST.pt` (or `net_weights_LAST.pt`)
exists in the output dir, the loop loads it — restoring model weights, optimizer,
scheduler, epoch counter, and best-F1 — then continues training from the next epoch.
Checkpoints are always saved/loaded **unwrapped** (no `module.` prefix), so resume works
correctly even with `DataParallel` across 2 GPUs. To force a fresh run, delete
`train_results/net_weights_*.pt` first (or change `cfg.train.out_dir`).

**Per-epoch full-slice inline visualization:** after each epoch's validation, the loop
loads one **full validation tomogram**, tiles the model across the entire central XY
slice, and renders a side-by-side `tomogram | GROUND TRUTH overlay | PREDICTION overlay`
on the **full slice** (same style as section 4a/4b), displayed inline under this training
cell (plus saved to `train_results/viz/epoch_NNN.png`).

Edit hyperparameters in the cell below. Set `epochs=2` for a smoke test.

In [ ]:
# --- Hyperparameters (edit freely) ---
cfg = Config(
    data=DataCfg(
        copick_config=config_path,
        input_root=INPUT, output_root=WORK,
        voxel_size=10, tomo_algorithm="denoised", tomo_type="denoised",
        target_name="pytargets", target_user_id="pytorch", target_session_id="0",
        dim_in=72, l_rnd=15, background_ratio=0.30,
        sample_size=5, n_sub_epoch=10,
    ),
    model=ModelCfg(name="res_unet", filters=[48, 64, 128], dropout=0.0,
                   in_channels=1, n_class=8),
    loss=LossCfg(type="ce_tversky", alpha=0.3, beta=0.7, gamma=2.0,
                 ce_weight=1.0, tversky_weight=1.0, smooth=1e-3,
                 class_weights="inverse", ignore_index=-1,
                 # classes that carry target signal (bg + 6 particles);
                 # membrane (class 7) excluded until explicitly trained
                 active_classes=[0, 1, 2, 3, 4, 5, 6]),
    train=TrainCfg(
        epochs=70, steps_per_epoch=150, batch_size=8, steps_per_valid=20,
        # n_workers=0 avoids duplicating the in-RAM tomogram pool across
        # forked workers (which would multiply resident memory ~4x and OOM on
        # Kaggle). Patches are already RAM-resident, so worker parallelism
        # buys little here.
        n_workers=0,
        vol_eval_every=10,  # run full-volume inference every 10 epochs
        pin_memory=True,
        optimizer="adamw", lr=1e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
        scheduler="cosine", warmup_epochs=3, min_lr=1e-6,
        amp=True, grad_clip=0.0,
        out_dir="/kaggle/working/train_results", save_every=10, resume=None, seed=42,
    ),
    inference=InferenceCfg(
        patch_size=72, overlap=55, pcrop=25, batch_patches=4, amp=True,
        write_scoremap=False, scoremap_name="pyscoremap",
        segmentation_name="pysegmentation", user_id="pytorch", session_id="0",
        out_overlay="/kaggle/working/predictions",
    ),
    localize=LocalizeCfg(
        min_protein_size=0.8, write_copick_picks=True, write_csv=False,
        csv_path="/kaggle/working/submission.csv", picks_user_id="pytorch", picks_session_id="0",
    ),
)
print("Config ready:")
print("  epochs=", cfg.train.epochs, "batch/GPU=", cfg.train.batch_size,
      "filters=", cfg.model.filters, "patch=", cfg.data.dim_in, "amp=", cfg.train.amp)
print("  active_classes=", cfg.loss.active_classes, "n_workers=", cfg.train.n_workers)


In [ ]:
%%time
best_ckpt = train(cfg)
print("Best checkpoint:", best_ckpt)

### 5a. Training curves

Per-epoch train/val loss and validation macro-F1, saved to `history.json` each epoch.
The per-epoch full-slice side-by-side figures are displayed inline during training (see
the training cell output above) and saved as PNGs under `train_results/viz/`.

In [ ]:
# Plot the training history saved by the loop
import json
from pathlib import Path
hist_path = Path(cfg.train.out_dir) / "history.json"
if hist_path.exists():
    h = json.loads(hist_path.read_text())
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(h.get("train_loss", []), label="train", lw=2)
    ax[0].plot(h.get("val_loss", []), label="val", lw=2)
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(h.get("val_macro_f1", []), label="macroF1", color="green", lw=2)
    ax[1].plot(h.get("val_macro_iou", []), label="macroIoU", color="purple", lw=2)
    ax[1].set_title("Validation metrics"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
    ax[2].plot(h.get("lr", []), color="darkorange", lw=2)
    ax[2].set_title("Learning rate"); ax[2].set_xlabel("epoch"); ax[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No history.json yet (training may still be running or just started).")

In [ ]:
# List the per-epoch full-slice side-by-side figures saved during training
from pathlib import Path
viz_dir = Path(cfg.train.out_dir) / "viz"
if viz_dir.exists():
    pngs = sorted(viz_dir.glob("epoch_*.png"))
    print(f"Per-epoch visualization PNGs ({len(pngs)}):")
    for p in pngs:
        print(f"  {p}  ({p.stat().st_size/1e3:.0f} KB)")
else:
    print("No viz/ directory yet (training hasn't completed an epoch).")

## 6. Stage 3 — Run inference (writes raw segmentations)

Sliding-window inference with cosine overlap blending. Writes a per-class **labelmap**
(argmax) as an OME-Zarr segmentation named `pysegmentation` into the overlay, and
optionally the full **probability scoremap** (`pyscoremap`) if `write_scoremap=True`.

These raw segmentations are the primary deliverable for your own postprocessing.

In [ ]:
cfg.inference.write_scoremap = False   # set True to also dump per-class probabilities (large)
weights = "/kaggle/working/train_results/net_weights_BEST.pt"
# Segment a few runs first to sanity-check, then set tomo_ids=None for all
tomo_ids = None
run_inference(cfg, weights, tomo_ids=tomo_ids)
print("Segmentation complete.")

### 6a. Prediction visualization — raw segmentations

Annotated overlays of the predicted labelmap on the tomogram, plus a z-slice montage
and orthogonal views, so you can directly compare against the ground-truth targets above.

In [ ]:
# Predicted segmentation vs tomogram — annotated overlay at mid-z with legend
import numpy as np, matplotlib.pyplot as plt
tid = (tomo_ids or runs)[0]
seg = get_segmentation(config_path, tid, name="pysegmentation", user_id="pytorch", session_id="0")[:]
z = tomo_vol.shape[0] // 2
fig, ax = plt.subplots(figsize=(9, 9))
overlay_slice(tomo_vol[z], seg[z], alpha=0.55, ax=ax,
              title=f"{tid} — PREDICTED segmentation overlay z={z}")
legend_for_labels(np.unique(seg), ax=ax)
plt.tight_layout(); plt.show()
print("Predicted labels:", np.unique(seg))

In [ ]:
# Predicted segmentation — z-slice montage with overlays
fig = montage(tomo_vol, labels=seg, n_slices=8, axis=0,
             suptitle=f"{tid} — PREDICTED segmentation overlay (z-slices)")
plt.show()

In [ ]:
# Predicted segmentation — orthogonal views
fig = orthogonal_views(tomo_vol, labels=seg, alpha=0.55,
                       suptitle=f"{tid} — PREDICTED segmentation (orthogonal)")
plt.show()

In [ ]:
# Per-class voxel breakdown of the prediction vs the target
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
class_breakdown(seg, ax=axes[0], title=f"{tid} — PREDICTED voxel counts")
class_breakdown(tgt, ax=axes[1], title=f"{tid} — TARGET voxel counts")
plt.tight_layout(); plt.show()

### 6b. Prediction vs ground-truth side-by-side

Direct comparison of the model's segmentation against the sphere targets, slice by slice.
Use this to spot which classes are under/over-predicted and where false positives appear.

In [ ]:
# Side-by-side: tomogram | target | prediction, at mid-z
z = tomo_vol.shape[0] // 2
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(tomo_vol[z], cmap="gray"); axes[0].set_title(f"tomogram z={z}"); axes[0].axis("off")
overlay_slice(tomo_vol[z], tgt[z], alpha=0.55, ax=axes[1], title=f"TARGET z={z}")
legend_for_labels(np.unique(tgt), ax=axes[1])
overlay_slice(tomo_vol[z], seg[z], alpha=0.55, ax=axes[2], title=f"PREDICTION z={z}")
legend_for_labels(np.unique(seg), ax=axes[2])
plt.tight_layout(); plt.show()

In [ ]:
# Difference map: prediction vs target (0=agree, +/- = over/under-predict)
diff = seg.astype(np.int16) - tgt.astype(np.int16)
fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(diff[z], cmap="RdBu", vmin=-6, vmax=6)
ax.set_title(f"prediction - target  (z={z})\nblue=pred<gt, red=pred>gt, white=agree")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="label(pred) - label(target)")
plt.tight_layout(); plt.show()
print("Agreement fraction at z=%d: %.3f" % (z, float((diff[z]==0).mean())))

## 7. Stage 4 (optional) — Localize particles from segmentations

Converts each class's connected components to (x,y,z) centers and writes them as
copick pick JSONs into `/kaggle/working/predictions`. Skip if you only want raw segmentations.

In [ ]:
cfg.inference.out_overlay = "/kaggle/working/predictions"
localize_segmentations(cfg, tomo_ids=None)
print("Localization complete. Picks written to /kaggle/working/predictions")

## 8. Artifacts

Kaggle automatically persists `/kaggle/working`. Key artifacts:
- `train_results/net_weights_BEST.pt` — best model checkpoint (auto-loaded on resume)
- `train_results/net_weights_LAST.pt` — latest checkpoint (fallback for resume)
- `train_results/config_used.yaml` — the exact config used
- `train_results/history.json` — per-epoch metrics for plotting
- `train_results/tensorboard_logs/` — TB event files
- `train_results/viz/epoch_NNN.png` — per-epoch full-slice side-by-side prediction figures
- copick overlay segmentations named `pysegmentation` (raw labelmaps)

In [ ]:
from pathlib import Path
print("Artifacts in /kaggle/working:")
for p in sorted(Path("/kaggle/working").rglob("*")):
    if p.is_file() and ".zarr" not in str(p) and p.stat().st_size > 0:
        print(f"  {p}  ({p.stat().st_size/1e6:.2f} MB)")